In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, make_scorer, balanced_accuracy_score, roc_auc_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, SelectFromModel
import time
import warnings
warnings.filterwarnings('ignore')
 
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from boruta import BorutaPy
import copy

from sklearn.base import BaseEstimator, TransformerMixin
 
from xgboost import XGBClassifier

In [3]:
df=pd.read_csv("/kaggle/input/datasets/inesbenlatreche/colon-cancer/Alot et al.csv")
print(df.head())
X = df.drop('Class', axis=1).values
y = (df['Class'] == 'positive').astype(int).values

   attribute1  attribute2  attribute3  attribute4  attribute5  attribute6  \
0   8589.4163   5468.2409   4263.4075   4064.9357   1997.8929   5282.3250   
1   9164.2537   6719.5295   4883.4487   3718.1589   2015.2214   5569.9071   
2   3825.7050   6970.3614   5369.9688   4705.6500   1166.5536   1572.1679   
3   6246.4487   7823.5341   5955.8350   3975.5643   2002.6131   2130.5429   
4   3230.3287   3694.4500   3400.7400   3463.5857   2181.4202   2922.7821   

   attribute7  attribute8  attribute9  attribute10  ...  attribute1992  \
0   2169.7200   2773.4212   7526.3862    4607.6762  ...       67.56125   
1   3849.0588   2793.3875   7017.7338    4802.2524  ...       92.23875   
2   1325.4025   1472.2587   3296.9512    2786.5821  ...       82.71500   
3   1531.1425   1714.6312   3869.7850    4989.4071  ...       41.68375   
4   2069.2463   2948.5750   3303.3712    3109.4131  ...       76.60375   

   attribute1993  attribute1994  attribute1995  attribute1996  attribute1997  \
0      259.9

In [4]:
K_FEATURES = 50          # number of features to select (tune this)
CV_FOLDS   = 5           # stratified k-fold (5 is safe for 62 samples)
SMOTE_K    = 3           # conservative k_neighbors for small minority class
 
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
 
scoring = {
    'balanced_acc': make_scorer(balanced_accuracy_score),
    'f1':           make_scorer(f1_score),
    'roc_auc': 'roc_auc',
}

In [5]:
# classifiers
classifiers = {
    'KNN':               KNeighborsClassifier(n_neighbors=5),
    'SVM':               SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),
    'Naive Bayes':       GaussianNB(),
    'Decision Tree':     DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest':     RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Logistic Reg':      LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
}

# Feature Selection

In [6]:
feature_selectors = {
    'ANOVA F-test':       SelectKBest(f_classif,           k=K_FEATURES),
    'Mutual Info':        SelectKBest(mutual_info_classif, k=K_FEATURES),
    'No Selection':       'passthrough',   # baseline: use all 2001 features
}

In [7]:
results = []
 
smote = SMOTE(random_state=42, k_neighbors=SMOTE_K)
 
for fs_name, selector in feature_selectors.items():
    for clf_name, clf in classifiers.items():
 
        # Naive Bayes and KNN don't support predict_proba natively after SMOTE
        # for roc_auc we need probability estimates — skip roc_auc for NB/KNN
        # (GaussianNB does support predict_proba, KNN also does — so all fine)
 
        steps = [
            ('smote',  smote),
            ('scaler', StandardScaler()),
        ]
        if selector != 'passthrough':
            steps.append(('selector', selector))
        steps.append(('clf', clf))
 
        pipeline = Pipeline(steps)
 
        try:
            scores = cross_validate(
                pipeline, X, y,
                cv=cv,
                scoring=scoring,
                return_train_score=False,
                n_jobs=-1
            )
            results.append({
                'Feature Selection': fs_name,
                'Classifier':        clf_name,
                'Balanced Acc':      np.mean(scores['test_balanced_acc']),
                'F1':                np.mean(scores['test_f1']),
                'ROC-AUC':           np.mean(scores['test_roc_auc']),
                'Balanced Acc Std':  np.std(scores['test_balanced_acc']),
            })
            print(f"  ✓  {fs_name:<18} | {clf_name:<18} | "
                  f"BalAcc={np.mean(scores['test_balanced_acc']):.3f} "
                  f"F1={np.mean(scores['test_f1']):.3f} "
                  f"AUC={np.mean(scores['test_roc_auc']):.3f}")
 
        except Exception as e:
            print(f"  ✗  {fs_name:<18} | {clf_name:<18} | ERROR: {e}")
            results.append({
                'Feature Selection': fs_name,
                'Classifier':        clf_name,
                'Balanced Acc': np.nan, 'F1': np.nan,
                'ROC-AUC': np.nan, 'Balanced Acc Std': np.nan,
            })

  ✓  ANOVA F-test       | KNN                | BalAcc=0.868 F1=0.824 AUC=0.889
  ✓  ANOVA F-test       | SVM                | BalAcc=0.835 F1=0.782 AUC=0.882
  ✓  ANOVA F-test       | Naive Bayes        | BalAcc=0.797 F1=0.744 AUC=0.896
  ✓  ANOVA F-test       | Decision Tree      | BalAcc=0.665 F1=0.550 AUC=0.665
  ✓  ANOVA F-test       | Random Forest      | BalAcc=0.815 F1=0.760 AUC=0.917
  ✓  ANOVA F-test       | Logistic Reg       | BalAcc=0.820 F1=0.766 AUC=0.911
  ✓  Mutual Info        | KNN                | BalAcc=0.838 F1=0.776 AUC=0.857
  ✓  Mutual Info        | SVM                | BalAcc=0.835 F1=0.781 AUC=0.887
  ✓  Mutual Info        | Naive Bayes        | BalAcc=0.785 F1=0.731 AUC=0.882
  ✓  Mutual Info        | Decision Tree      | BalAcc=0.600 F1=0.400 AUC=0.600
  ✓  Mutual Info        | Random Forest      | BalAcc=0.855 F1=0.808 AUC=0.916
  ✓  Mutual Info        | Logistic Reg       | BalAcc=0.797 F1=0.732 AUC=0.895
  ✓  No Selection       | KNN                | BalAc

In [9]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values('Balanced Acc', ascending=False).reset_index(drop=True)
 
print("\n" + "="*80)
print("RESULTS (sorted by Balanced Accuracy)")
print("="*80)
print(df_results.to_string(index=False, float_format='{:.3f}'.format))


RESULTS (sorted by Balanced Accuracy)
Feature Selection    Classifier  Balanced Acc    F1  ROC-AUC  Balanced Acc Std
     ANOVA F-test           KNN         0.868 0.824    0.889             0.070
      Mutual Info Random Forest         0.855 0.808    0.916             0.100
     No Selection  Logistic Reg         0.847 0.798    0.869             0.046
      Mutual Info           KNN         0.838 0.776    0.857             0.085
     ANOVA F-test           SVM         0.835 0.782    0.882             0.076
      Mutual Info           SVM         0.835 0.781    0.887             0.060
     ANOVA F-test  Logistic Reg         0.820 0.766    0.911             0.076
     ANOVA F-test Random Forest         0.815 0.760    0.917             0.099
     No Selection           KNN         0.800 0.739    0.856             0.073
     ANOVA F-test   Naive Bayes         0.797 0.744    0.896             0.050
     No Selection Random Forest         0.797 0.723    0.922             0.093
      Mutual 

In [10]:
# ── 6. Best combo ────────────────────────────────────────────────────────────
best = df_results.iloc[0]
print(f"\n🏆 Best combination:")
print(f"   Feature Selection : {best['Feature Selection']}")
print(f"   Classifier        : {best['Classifier']}")
print(f"   Balanced Accuracy : {best['Balanced Acc']:.3f} ± {best['Balanced Acc Std']:.3f}")
print(f"   F1 Score          : {best['F1']:.3f}")
print(f"   ROC-AUC           : {best['ROC-AUC']:.3f}")
 
# Optional: save results
df_results.to_csv('benchmark_results.csv', index=False)
print("\n📄 Full results saved to benchmark_results.csv")
 


🏆 Best combination:
   Feature Selection : ANOVA F-test
   Classifier        : KNN
   Balanced Accuracy : 0.868 ± 0.070
   F1 Score          : 0.824
   ROC-AUC           : 0.889

📄 Full results saved to benchmark_results.csv


### feature inspection:
see what features are selected by the two filter methods 

In [11]:
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
# ── 1. Scale first (same as in the benchmark pipeline) ───────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 
# ── 2. Fit both selectors ─────────────────────────────────────────────────────
anova = SelectKBest(f_classif, k=K_FEATURES).fit(X_scaled, y)
mi    = SelectKBest(mutual_info_classif, k=K_FEATURES).fit(X_scaled, y)
 
anova_mask = anova.get_support()
mi_mask    = mi.get_support()
 
anova_features = set(np.array(feature_names)[anova_mask])
mi_features    = set(np.array(feature_names)[mi_mask])

In [12]:
# ── 3. Overlap analysis ───────────────────────────────────────────────────────
common   = anova_features & mi_features
anova_only = anova_features - mi_features
mi_only    = mi_features - anova_features
 
print("=" * 55)
print(f"  Total features selected (K)  : {K_FEATURES}")
print(f"  Selected by ANOVA only       : {len(anova_only)}")
print(f"  Selected by Mutual Info only : {len(mi_only)}")
print(f"  Selected by BOTH             : {len(common)}")
print(f"  Overlap ratio                : {len(common)/K_FEATURES*100:.1f}%")
print("=" * 55)

  Total features selected (K)  : 50
  Selected by ANOVA only       : 30
  Selected by Mutual Info only : 30
  Selected by BOTH             : 20
  Overlap ratio                : 40.0%


In [13]:
# ── 4. Score tables ───────────────────────────────────────────────────────────
anova_scores = pd.DataFrame({
    'feature':     np.array(feature_names)[anova_mask],
    'anova_score': anova.scores_[anova_mask],
    'anova_pval':  anova.pvalues_[anova_mask],
}).sort_values('anova_score', ascending=False).reset_index(drop=True)
 
mi_scores = pd.DataFrame({
    'feature':   np.array(feature_names)[mi_mask],
    'mi_score':  mi.scores_[mi_mask],
}).sort_values('mi_score', ascending=False).reset_index(drop=True)
 
print("\nTop 10 features by ANOVA F-score:")
print(anova_scores.head(10).to_string(index=False))
 
print("\nTop 10 features by Mutual Info score:")
print(mi_scores.head(10).to_string(index=False))


Top 10 features by ANOVA F-score:
     feature  anova_score   anova_pval
 feature_248    39.812669 3.711979e-08
 feature_764    33.149759 3.108866e-07
 feature_492    32.015918 4.535944e-07
feature_1422    31.760616 4.942077e-07
 feature_244    30.949993 6.499838e-07
 feature_266    29.643523 1.016521e-06
 feature_376    25.339396 4.672006e-06
 feature_821    24.810264 5.668504e-06
feature_1891    20.539119 2.841617e-05
feature_1771    19.443601 4.364999e-05

Top 10 features by Mutual Info score:
     feature  mi_score
 feature_624  0.295897
feature_1670  0.266871
feature_1866  0.266279
 feature_257  0.253407
 feature_512  0.245939
feature_1292  0.234923
 feature_681  0.229420
feature_1770  0.229402
feature_1934  0.228197
feature_1762  0.224972


In [15]:
# ── 5. Build combined CSV ─────────────────────────────────────────────────────
all_features = sorted(anova_features | mi_features)
 
df_export = pd.DataFrame({'feature': all_features})
df_export['selected_by_ANOVA']  = df_export['feature'].isin(anova_features)
df_export['selected_by_MI']     = df_export['feature'].isin(mi_features)
df_export['selected_by_BOTH']   = df_export['feature'].isin(common)
 
# Merge scores
df_export = df_export.merge(anova_scores[['feature','anova_score','anova_pval']], on='feature', how='left')
df_export = df_export.merge(mi_scores[['feature','mi_score']], on='feature', how='left')
 
df_export = df_export.sort_values(['selected_by_BOTH', 'anova_score'],
                                   ascending=[False, False]).reset_index(drop=True)
 
output_path = 'selected_features.csv'
df_export.to_csv(output_path, index=False)
print(f"\n📄 CSV saved to: {output_path}")
print(f"   Columns: feature | selected_by_ANOVA | selected_by_MI | selected_by_BOTH | anova_score | anova_pval | mi_score")
print(f"   Rows: {len(df_export)} features total ({len(common)} shared, {len(anova_only)} ANOVA-only, {len(mi_only)} MI-only)")


📄 CSV saved to: selected_features.csv
   Columns: feature | selected_by_ANOVA | selected_by_MI | selected_by_BOTH | anova_score | anova_pval | mi_score
   Rows: 80 features total (20 shared, 30 ANOVA-only, 30 MI-only)






# Wrapper methods

In [18]:
# ── 3. Wrapper selectors ──────────────────────────────────────────────────────
# RFE needs a model with coef_ → LinearSVC or LogisticRegression
# step=0.1 means remove 10% of features per iteration (faster than step=1)
rfe_svm = RFE(
    estimator=LinearSVC(C=1.0, max_iter=5000, random_state=42),
    n_features_to_select=K_FEATURES,
    step=0.1
)
 
rfe_lr = RFE(
    estimator=LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42),
    n_features_to_select=K_FEATURES,
    step=0.1
)
 
# Boruta uses Random Forest internally and selects features that outperform
# shuffled shadow features. It determines K automatically.
# perc=90 (slightly lenient) works better on small datasets than perc=100
boruta_selector = BorutaPy(
    estimator=RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                     random_state=42, n_jobs=-1),
    n_estimators='auto',
    perc=90,
    max_iter=50,        # keep runtime manageable
    random_state=42
)
 
feature_selectors = {
    'RFE-SVM':    rfe_svm,
    'RFE-LogReg': rfe_lr,
    'Boruta':     boruta_selector,
}

In [20]:
# ── 4. Helper: clone-safe selector wrapper ────────────────────────────────────
# RFE and Boruta don't survive sklearn's clone() inside cross_validate
# because they hold state. We wrap them in a FunctionTransformer-style
# custom transformer that re-fits from scratch each fold.
 
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
 
class WrapperSelector(BaseEstimator, TransformerMixin):
    """Wraps RFE / Boruta so they can be cloned safely inside a pipeline."""
    def __init__(self, selector):
        self.selector = selector
 
    def fit(self, X, y=None):
        import copy
        self.selector_ = copy.deepcopy(self.selector)
        self.selector_.fit(X, y)
        return self
 
    def transform(self, X):
        return self.selector_.transform(X)
 
    def get_support(self):
        return self.selector_.support_

In [21]:
# ── 5. Run benchmark ──────────────────────────────────────────────────────────
results     = []
timing_rows = []
 
print(f"{'FS Method':<14} | {'Classifier':<14} | {'BalAcc':>7} {'F1':>6} {'AUC':>6} | {'Time':>8}")
print("-" * 65)
 
for fs_name, selector in feature_selectors.items():
    for clf_name, clf in classifiers.items():
 
        steps = [
            ('smote',    smote),
            ('scaler',   StandardScaler()),
            ('selector', WrapperSelector(selector)),
            ('clf',      clf),
        ]
        pipeline = Pipeline(steps)
 
        t_start = time.time()
        try:
            scores = cross_validate(
                pipeline, X, y,
                cv=cv,
                scoring=scoring,
                return_train_score=False,
                n_jobs=1,   # keep n_jobs=1 — wrapper selectors are not thread-safe
                error_score='raise'
            )
            elapsed = time.time() - t_start
 
            bal_acc = np.mean(scores['test_balanced_acc'])
            f1      = np.mean(scores['test_f1'])
            auc     = np.mean(scores['test_roc_auc'])
            std     = np.std(scores['test_balanced_acc'])
 
            results.append({
                'Feature Selection': fs_name,
                'Classifier':        clf_name,
                'Balanced Acc':      bal_acc,
                'F1':                f1,
                'ROC-AUC':           auc,
                'Balanced Acc Std':  std,
                'Time (s)':          round(elapsed, 1),
            })
            print(f"  ✓  {fs_name:<12} | {clf_name:<14} | "
                  f"{bal_acc:.3f}  {f1:.3f}  {auc:.3f} | {elapsed:>6.1f}s")
 
        except Exception as e:
            elapsed = time.time() - t_start
            short_err = str(e).split('\n')[0][:60]
            print(f"  ✗  {fs_name:<12} | {clf_name:<14} | ERROR: {short_err}")
            results.append({
                'Feature Selection': fs_name,
                'Classifier':        clf_name,
                'Balanced Acc': np.nan, 'F1': np.nan,
                'ROC-AUC': np.nan, 'Balanced Acc Std': np.nan,
                'Time (s)': round(elapsed, 1),
            })

FS Method      | Classifier     |  BalAcc     F1    AUC |     Time
-----------------------------------------------------------------
  ✓  RFE-SVM      | KNN            | 0.812  0.749  0.876 |   11.3s
  ✓  RFE-SVM      | SVM            | 0.823  0.760  0.875 |   11.6s
  ✓  RFE-SVM      | Naive Bayes    | 0.785  0.718  0.920 |   11.1s
  ✓  RFE-SVM      | Decision Tree  | 0.620  0.479  0.620 |   11.1s
  ✓  RFE-SVM      | Random Forest  | 0.828  0.775  0.891 |   11.9s
  ✓  RFE-SVM      | Logistic Reg   | 0.830  0.766  0.926 |   11.3s
  ✓  RFE-LogReg   | KNN            | 0.830  0.766  0.861 |    1.1s
  ✓  RFE-LogReg   | SVM            | 0.778  0.702  0.886 |    1.1s
  ✓  RFE-LogReg   | Naive Bayes    | 0.810  0.751  0.869 |    1.2s
  ✓  RFE-LogReg   | Decision Tree  | 0.657  0.503  0.657 |    1.1s
  ✓  RFE-LogReg   | Random Forest  | 0.843  0.795  0.909 |    2.1s
  ✓  RFE-LogReg   | Logistic Reg   | 0.802  0.735  0.879 |    1.1s
  ✓  Boruta       | KNN            | 0.830  0.768  0.891 |  215

In [18]:
# ── 6. Results table ──────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
df_results = df_results.sort_values('Balanced Acc', ascending=False).reset_index(drop=True)
 
print("\n" + "=" * 80)
print("WRAPPER RESULTS (sorted by Balanced Accuracy)")
print("=" * 80)
print(df_results.to_string(index=False, float_format='{:.3f}'.format))


WRAPPER RESULTS (sorted by Balanced Accuracy)
Feature Selection    Classifier  Balanced Acc    F1  ROC-AUC  Balanced Acc Std  Time (s)
       RFE-LogReg Random Forest         0.843 0.795    0.909             0.088     2.000
           Boruta           SVM         0.835 0.781    0.881             0.060   212.700
           Boruta  Logistic Reg         0.835 0.781    0.907             0.060   209.700
       RFE-LogReg           KNN         0.830 0.766    0.861             0.117     1.000
           Boruta           KNN         0.830 0.768    0.891             0.073   213.100
          RFE-SVM  Logistic Reg         0.830 0.766    0.926             0.117    11.000
          RFE-SVM Random Forest         0.828 0.775    0.891             0.097    11.700
          RFE-SVM           SVM         0.823 0.760    0.875             0.128    11.100
           Boruta Random Forest         0.823 0.760    0.893             0.094   210.300
          RFE-SVM           KNN         0.812 0.749    0.876   

In [19]:
# ── 7. Timing summary ─────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("TIMING SUMMARY (total time per feature selection method)")
print("=" * 80)
timing_summary = (
    df_results.groupby('Feature Selection')['Time (s)']
    .agg(Total_s='sum', Per_clf_avg='mean')
    .reset_index()
    .sort_values('Total_s')
)
print(timing_summary.to_string(index=False, float_format='{:.1f}'.format))


TIMING SUMMARY (total time per feature selection method)
Feature Selection  Total_s  Per_clf_avg
       RFE-LogReg      7.2          1.2
          RFE-SVM     66.8         11.1
           Boruta   1267.9        211.3


In [20]:
# ── 8. Best combo ─────────────────────────────────────────────────────────────
best = df_results.dropna(subset=['Balanced Acc']).iloc[0]
print(f"\n🏆 Best wrapper combination:")
print(f"   Feature Selection : {best['Feature Selection']}")
print(f"   Classifier        : {best['Classifier']}")
print(f"   Balanced Accuracy : {best['Balanced Acc']:.3f} ± {best['Balanced Acc Std']:.3f}")
print(f"   F1 Score          : {best['F1']:.3f}")
print(f"   ROC-AUC           : {best['ROC-AUC']:.3f}")
print(f"   Time              : {best['Time (s)']:.1f}s")
 
df_results.to_csv('wrapper_results.csv', index=False)
print("\n📄 Full results saved to wrapper_results.csv")


🏆 Best wrapper combination:
   Feature Selection : RFE-LogReg
   Classifier        : Random Forest
   Balanced Accuracy : 0.843 ± 0.088
   F1 Score          : 0.795
   ROC-AUC           : 0.909
   Time              : 2.0s

📄 Full results saved to wrapper_results.csv


In [21]:
# ── 2. Fit each wrapper selector ──────────────────────────────────────────────
 
# RFE-SVM
print("Fitting RFE-SVM...")
rfe_svm = RFE(
    estimator=LinearSVC(C=1.0, max_iter=5000, random_state=42),
    n_features_to_select=K_FEATURES,
    step=0.1
)
rfe_svm.fit(X_scaled, y)
rfe_svm_mask     = rfe_svm.support_
rfe_svm_ranking  = rfe_svm.ranking_   # 1 = selected, higher = eliminated earlier
 
# RFE-LogReg
print("Fitting RFE-LogReg...")
rfe_lr = RFE(
    estimator=LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42),
    n_features_to_select=K_FEATURES,
    step=0.1
)
rfe_lr.fit(X_scaled, y)
rfe_lr_mask    = rfe_lr.support_
rfe_lr_ranking = rfe_lr.ranking_
 
# Boruta
print("Fitting Boruta (may take a moment)...")
boruta = BorutaPy(
    estimator=RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                     random_state=42, n_jobs=-1),
    n_estimators='auto',
    perc=90,
    max_iter=50,
    random_state=42
)
boruta.fit(X_scaled, y)
boruta_mask    = boruta.support_
boruta_ranking = boruta.ranking_   # 1 = confirmed, higher = rejected earlier

Fitting RFE-SVM...
Fitting RFE-LogReg...
Fitting Boruta (may take a moment)...


In [22]:
# ── 3. Feature sets ───────────────────────────────────────────────────────────
feature_arr = np.array(feature_names)
 
rfe_svm_features = set(feature_arr[rfe_svm_mask])
rfe_lr_features  = set(feature_arr[rfe_lr_mask])
boruta_features  = set(feature_arr[boruta_mask])
 
all_selected = rfe_svm_features | rfe_lr_features | boruta_features
common_all   = rfe_svm_features & rfe_lr_features & boruta_features
common_svm_lr    = rfe_svm_features & rfe_lr_features - common_all
common_svm_bor   = rfe_svm_features & boruta_features - common_all
common_lr_bor    = rfe_lr_features  & boruta_features - common_all

In [23]:
# ── 4. Overlap summary ────────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"  RFE-SVM selected      : {rfe_svm_mask.sum()}")
print(f"  RFE-LogReg selected   : {rfe_lr_mask.sum()}")
print(f"  Boruta selected       : {boruta_mask.sum()}")
print(f"  Union (any method)    : {len(all_selected)}")
print(f"  Common (all 3)        : {len(common_all)}")
print(f"  Common (SVM + LR)     : {len(rfe_svm_features & rfe_lr_features)}")
print(f"  Common (SVM + Boruta) : {len(rfe_svm_features & boruta_features)}")
print(f"  Common (LR + Boruta)  : {len(rfe_lr_features  & boruta_features)}")
print("="*55)


  RFE-SVM selected      : 50
  RFE-LogReg selected   : 50
  Boruta selected       : 110
  Union (any method)    : 161
  Common (all 3)        : 12
  Common (SVM + LR)     : 30
  Common (SVM + Boruta) : 13
  Common (LR + Boruta)  : 18


In [24]:
# ── 5. Build combined CSV ─────────────────────────────────────────────────────
all_feat_sorted = sorted(all_selected)
 
df_export = pd.DataFrame({'feature': all_feat_sorted})
df_export['selected_by_RFE_SVM']   = df_export['feature'].isin(rfe_svm_features)
df_export['selected_by_RFE_LogReg']= df_export['feature'].isin(rfe_lr_features)
df_export['selected_by_Boruta']    = df_export['feature'].isin(boruta_features)
df_export['selected_by_ALL']       = df_export['feature'].isin(common_all)
 
# How many methods selected each feature (0-3)
df_export['n_methods'] = (
    df_export['selected_by_RFE_SVM'].astype(int) +
    df_export['selected_by_RFE_LogReg'].astype(int) +
    df_export['selected_by_Boruta'].astype(int)
)
 
# Merge rankings (lower = more important)
rfe_svm_rank_df = pd.DataFrame({'feature': feature_arr, 'rfe_svm_ranking': rfe_svm_ranking})
rfe_lr_rank_df  = pd.DataFrame({'feature': feature_arr, 'rfe_lr_ranking':  rfe_lr_ranking})
boruta_rank_df  = pd.DataFrame({'feature': feature_arr, 'boruta_ranking':  boruta_ranking})
 
df_export = df_export.merge(rfe_svm_rank_df, on='feature', how='left')
df_export = df_export.merge(rfe_lr_rank_df,  on='feature', how='left')
df_export = df_export.merge(boruta_rank_df,  on='feature', how='left')
 
# Sort: features selected by most methods first, then by RFE-SVM ranking
df_export = df_export.sort_values(
    ['n_methods', 'rfe_svm_ranking'],
    ascending=[False, True]
).reset_index(drop=True)
 
output_path = 'wrapper_selected_features.csv'
df_export.to_csv(output_path, index=False)
 
print(f"\n📄 CSV saved to: {output_path}")
print(f"   Columns: feature | selected_by_RFE_SVM | selected_by_RFE_LogReg | "
      f"selected_by_Boruta | selected_by_ALL | n_methods | rfe_svm_ranking | rfe_lr_ranking | boruta_ranking")
print(f"   Rows: {len(df_export)} features "
      f"({len(common_all)} selected by all 3, {len(all_selected)} total union)")


📄 CSV saved to: wrapper_selected_features.csv
   Columns: feature | selected_by_RFE_SVM | selected_by_RFE_LogReg | selected_by_Boruta | selected_by_ALL | n_methods | rfe_svm_ranking | rfe_lr_ranking | boruta_ranking
   Rows: 161 features (12 selected by all 3, 161 total union)


# Embdedded 

In [25]:
# ── 3. Embedded selectors ─────────────────────────────────────────────────────
#
# All four use SelectFromModel — the embedded model is fit during the pipeline,
# its coef_ or feature_importances_ are read, and the top K features are kept.
# threshold=-np.inf + max_features=K guarantees exactly K features always selected.
 
# 3a. Lasso — L1 Logistic Regression
#     Shrinks weak feature log-odds coefficients to exactly 0 during fitting.
lasso_selector = SelectFromModel(
    LogisticRegression(
        penalty='l1', solver='liblinear',
        C=0.1, class_weight='balanced',
        max_iter=1000, random_state=42
    ),
    max_features=K_FEATURES,
    threshold=-np.inf
)
 
# 3b. ElasticNet — L1 + L2 mix via LogisticRegression
#     l1_ratio=0.5 means equal weight to both penalties.
#     Better than pure Lasso when features are correlated (common in omics data).
elasticnet_selector = SelectFromModel(
    LogisticRegression(
        penalty='elasticnet', solver='saga',
        l1_ratio=0.5, C=0.1,
        class_weight='balanced',
        max_iter=2000, random_state=42
    ),
    max_features=K_FEATURES,
    threshold=-np.inf
)
 
# 3c. Random Forest importances
#     Uses mean decrease in Gini impurity across all trees.
#     Captures non-linear relationships, no distribution assumptions.
rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    max_features=K_FEATURES,
    threshold=-np.inf
)
 
# 3d. XGBoost importances
#     Uses gain-based feature importance (how much each feature improves splits).
#     scale_pos_weight handles class imbalance natively in XGBoost.
scale_pos_weight = 40 / 22   # negative / positive count
 
xgb_selector = SelectFromModel(
    XGBClassifier(
        n_estimators=200,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ),
    max_features=K_FEATURES,
    threshold=-np.inf
)
 
feature_selectors = {
    'Lasso':       lasso_selector,
    'ElasticNet':  elasticnet_selector,
    'RF-Embed':    rf_selector,
    'XGBoost':     xgb_selector,
}

In [26]:
# ── 4. Clone-safe embedded selector wrapper ───────────────────────────────────
# SelectFromModel holds fitted state after the first fold — we deep-copy it
# before each fold so every fold starts fresh (same pattern as wrapper script).
 
class EmbeddedSelector(BaseEstimator, TransformerMixin):
    """Deep-copies the SelectFromModel each fold to prevent state leakage."""
    def __init__(self, selector):
        self.selector = selector
 
    def fit(self, X, y=None):
        self.selector_ = copy.deepcopy(self.selector)
        self.selector_.fit(X, y)
        return self
 
    def transform(self, X):
        return self.selector_.transform(X)
 
    def get_support(self):
        return self.selector_.get_support()

In [27]:
# ── 5. Run benchmark ──────────────────────────────────────────────────────────
results = []
 
print(f"{'FS Method':<12} | {'Classifier':<14} | {'BalAcc':>7} {'F1':>6} {'AUC':>6} | {'Time':>8}")
print("-" * 68)
 
for fs_name, selector in feature_selectors.items():
    for clf_name, clf in classifiers.items():
 
        steps = [
            ('smote',    smote),
            ('scaler',   StandardScaler()),
            ('selector', EmbeddedSelector(selector)),
            ('clf',      clf),
        ]
        pipeline = Pipeline(steps)
 
        t_start = time.time()
        try:
            scores = cross_validate(
                pipeline, X, y,
                cv=cv,
                scoring=scoring,
                return_train_score=False,
                n_jobs=1,          # n_jobs=1 for reproducible timing
                error_score='raise'
            )
            elapsed = time.time() - t_start
 
            bal_acc = np.mean(scores['test_balanced_acc'])
            f1      = np.mean(scores['test_f1'])
            auc     = np.mean(scores['test_roc_auc'])
            std     = np.std(scores['test_balanced_acc'])
 
            results.append({
                'Feature Selection': fs_name,
                'Classifier':        clf_name,
                'Balanced Acc':      bal_acc,
                'F1':                f1,
                'ROC-AUC':           auc,
                'Balanced Acc Std':  std,
                'Time (s)':          round(elapsed, 1),
            })
            print(f"  ✓  {fs_name:<10} | {clf_name:<14} | "
                  f"{bal_acc:.3f}  {f1:.3f}  {auc:.3f} | {elapsed:>6.1f}s")
 
        except Exception as e:
            elapsed = time.time() - t_start
            short_err = str(e).split('\n')[0][:60]
            print(f"  ✗  {fs_name:<10} | {clf_name:<14} | ERROR: {short_err}")
            results.append({
                'Feature Selection': fs_name,
                'Classifier':        clf_name,
                'Balanced Acc': np.nan, 'F1': np.nan,
                'ROC-AUC': np.nan, 'Balanced Acc Std': np.nan,
                'Time (s)': round(elapsed, 1),
            })

FS Method    | Classifier     |  BalAcc     F1    AUC |     Time
--------------------------------------------------------------------
  ✓  Lasso      | KNN            | 0.843  0.788  0.864 |    0.1s
  ✓  Lasso      | SVM            | 0.880  0.839  0.874 |    0.1s
  ✓  Lasso      | Naive Bayes    | 0.710  0.630  0.884 |    0.1s
  ✓  Lasso      | Decision Tree  | 0.698  0.628  0.698 |    0.1s
  ✓  Lasso      | Random Forest  | 0.835  0.781  0.890 |    0.8s
  ✓  Lasso      | Logistic Reg   | 0.872  0.831  0.884 |    0.2s
  ✓  ElasticNet | KNN            | 0.855  0.803  0.891 |   15.8s
  ✓  ElasticNet | SVM            | 0.847  0.799  0.899 |   15.7s
  ✓  ElasticNet | Naive Bayes    | 0.805  0.753  0.905 |   15.6s
  ✓  ElasticNet | Decision Tree  | 0.595  0.413  0.595 |   15.7s
  ✓  ElasticNet | Random Forest  | 0.835  0.783  0.892 |   16.3s
  ✓  ElasticNet | Logistic Reg   | 0.787  0.709  0.887 |   15.9s
  ✓  RF-Embed   | KNN            | 0.830  0.768  0.905 |    3.7s
  ✓  RF-Embed   | SVM

In [28]:
# ── 6. Results table ──────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
df_results = df_results.sort_values('Balanced Acc', ascending=False).reset_index(drop=True)
 
print("\n" + "=" * 80)
print("EMBEDDED RESULTS (sorted by Balanced Accuracy)")
print("=" * 80)
print(df_results.to_string(index=False, float_format='{:.3f}'.format))


EMBEDDED RESULTS (sorted by Balanced Accuracy)
Feature Selection    Classifier  Balanced Acc    F1  ROC-AUC  Balanced Acc Std  Time (s)
         RF-Embed  Logistic Reg         0.893 0.859    0.885             0.068     3.200
            Lasso           SVM         0.880 0.839    0.874             0.089     0.100
            Lasso  Logistic Reg         0.872 0.831    0.884             0.059     0.200
         RF-Embed   Naive Bayes         0.863 0.814    0.935             0.073     3.300
       ElasticNet           KNN         0.855 0.803    0.891             0.073    15.800
         RF-Embed Random Forest         0.847 0.806    0.898             0.100     4.000
       ElasticNet           SVM         0.847 0.799    0.899             0.066    15.700
            Lasso           KNN         0.843 0.788    0.864             0.124     0.100
          XGBoost           KNN         0.843 0.784    0.906             0.093     1.600
       ElasticNet Random Forest         0.835 0.783    0.892  

In [29]:
# ── 7. Timing summary ─────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("TIMING SUMMARY (total time per feature selection method)")
print("=" * 80)
timing_summary = (
    df_results.groupby('Feature Selection')['Time (s)']
    .agg(Total_s='sum', Per_clf_avg='mean')
    .reset_index()
    .sort_values('Total_s')
)
print(timing_summary.to_string(index=False, float_format='{:.1f}'.format))


TIMING SUMMARY (total time per feature selection method)
Feature Selection  Total_s  Per_clf_avg
            Lasso      1.4          0.2
          XGBoost      9.8          1.6
         RF-Embed     21.0          3.5
       ElasticNet     95.0         15.8


In [30]:
# ── 8. Best combo ─────────────────────────────────────────────────────────────
best = df_results.dropna(subset=['Balanced Acc']).iloc[0]
print(f"\n🏆 Best embedded combination:")
print(f"   Feature Selection : {best['Feature Selection']}")
print(f"   Classifier        : {best['Classifier']}")
print(f"   Balanced Accuracy : {best['Balanced Acc']:.3f} ± {best['Balanced Acc Std']:.3f}")
print(f"   F1 Score          : {best['F1']:.3f}")
print(f"   ROC-AUC           : {best['ROC-AUC']:.3f}")
print(f"   Time              : {best['Time (s)']:.1f}s")
 
df_results.to_csv('embedded_results.csv', index=False)
print("\n📄 Full results saved to embedded_results.csv")


🏆 Best embedded combination:
   Feature Selection : RF-Embed
   Classifier        : Logistic Reg
   Balanced Accuracy : 0.893 ± 0.068
   F1 Score          : 0.859
   ROC-AUC           : 0.885
   Time              : 3.2s

📄 Full results saved to embedded_results.csv


In [31]:
# Apply SMOTE + scaling once on full data (for feature inspection only, not CV)
X_res, y_res = smote.fit_resample(X, y)
X_scaled     = scaler.fit_transform(X_res)

In [32]:
# ── 2. Fit each embedded selector ─────────────────────────────────────────────
 
# Lasso
print("Fitting Lasso...")
lasso = SelectFromModel(
    LogisticRegression(penalty='l1', solver='liblinear', C=0.1,
                       class_weight='balanced', max_iter=1000, random_state=42),
    max_features=K_FEATURES, threshold=-np.inf
)
lasso.fit(X_scaled, y_res)
lasso_mask   = lasso.get_support()
lasso_scores = np.abs(lasso.estimator_.coef_[0])   # |coefficient| as importance
 
# ElasticNet
print("Fitting ElasticNet...")
enet = SelectFromModel(
    LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
                       C=0.1, class_weight='balanced', max_iter=2000, random_state=42),
    max_features=K_FEATURES, threshold=-np.inf
)
enet.fit(X_scaled, y_res)
enet_mask   = enet.get_support()
enet_scores = np.abs(enet.estimator_.coef_[0])
 
# Random Forest
print("Fitting RF-Embed...")
rf = SelectFromModel(
    RandomForestClassifier(n_estimators=200, class_weight='balanced',
                           random_state=42, n_jobs=-1),
    max_features=K_FEATURES, threshold=-np.inf
)
rf.fit(X_scaled, y_res)
rf_mask   = rf.get_support()
rf_scores = rf.estimator_.feature_importances_
 
# XGBoost
print("Fitting XGBoost...")
xgb = SelectFromModel(
    XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos_weight,
                  eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0),
    max_features=K_FEATURES, threshold=-np.inf
)
xgb.fit(X_scaled, y_res)
xgb_mask   = xgb.get_support()
xgb_scores = xgb.estimator_.feature_importances_

Fitting Lasso...
Fitting ElasticNet...
Fitting RF-Embed...
Fitting XGBoost...


In [33]:
# ── 3. Feature sets ───────────────────────────────────────────────────────────
feature_arr = np.array(feature_names)
 
lasso_features = set(feature_arr[lasso_mask])
enet_features  = set(feature_arr[enet_mask])
rf_features    = set(feature_arr[rf_mask])
xgb_features   = set(feature_arr[xgb_mask])
 
all_selected = lasso_features | enet_features | rf_features | xgb_features
common_all   = lasso_features & enet_features & rf_features & xgb_features

In [34]:
# ── 4. Overlap summary ────────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"  Lasso selected        : {lasso_mask.sum()}")
print(f"  ElasticNet selected   : {enet_mask.sum()}")
print(f"  RF-Embed selected     : {rf_mask.sum()}")
print(f"  XGBoost selected      : {xgb_mask.sum()}")
print(f"  Union (any method)    : {len(all_selected)}")
print(f"  Common (all 4)        : {len(common_all)}")
print(f"  Common (Lasso+ENet)   : {len(lasso_features & enet_features)}")
print(f"  Common (RF+XGB)       : {len(rf_features & xgb_features)}")
print("="*55)


  Lasso selected        : 50
  ElasticNet selected   : 50
  RF-Embed selected     : 50
  XGBoost selected      : 50
  Union (any method)    : 139
  Common (all 4)        : 2
  Common (Lasso+ENet)   : 16
  Common (RF+XGB)       : 12


In [35]:
# ── 5. Build combined CSV ─────────────────────────────────────────────────────
all_feat_sorted = sorted(all_selected)
 
df_export = pd.DataFrame({'feature': all_feat_sorted})
df_export['selected_by_Lasso']      = df_export['feature'].isin(lasso_features)
df_export['selected_by_ElasticNet'] = df_export['feature'].isin(enet_features)
df_export['selected_by_RF']         = df_export['feature'].isin(rf_features)
df_export['selected_by_XGBoost']    = df_export['feature'].isin(xgb_features)
df_export['selected_by_ALL']        = df_export['feature'].isin(common_all)
 
# How many methods selected each feature (0-4)
df_export['n_methods'] = (
    df_export['selected_by_Lasso'].astype(int) +
    df_export['selected_by_ElasticNet'].astype(int) +
    df_export['selected_by_RF'].astype(int) +
    df_export['selected_by_XGBoost'].astype(int)
)
 
# Merge importance scores per method
lasso_score_df = pd.DataFrame({'feature': feature_arr, 'lasso_coef':      lasso_scores})
enet_score_df  = pd.DataFrame({'feature': feature_arr, 'elasticnet_coef': enet_scores})
rf_score_df    = pd.DataFrame({'feature': feature_arr, 'rf_importance':   rf_scores})
xgb_score_df   = pd.DataFrame({'feature': feature_arr, 'xgb_importance':  xgb_scores})
 
df_export = df_export.merge(lasso_score_df, on='feature', how='left')
df_export = df_export.merge(enet_score_df,  on='feature', how='left')
df_export = df_export.merge(rf_score_df,    on='feature', how='left')
df_export = df_export.merge(xgb_score_df,   on='feature', how='left')
 
# Sort: features selected by most methods first, then by XGBoost importance
df_export = df_export.sort_values(
    ['n_methods', 'xgb_importance'],
    ascending=[False, False]
).reset_index(drop=True)
 
output_path = 'embedded_selected_features.csv'
df_export.to_csv(output_path, index=False)
 
print(f"\n📄 CSV saved to: {output_path}")
print(f"   Columns: feature | selected_by_Lasso | selected_by_ElasticNet | "
      f"selected_by_RF | selected_by_XGBoost | selected_by_ALL | n_methods | "
      f"lasso_coef | elasticnet_coef | rf_importance | xgb_importance")
print(f"   Rows: {len(df_export)} features "
      f"({len(common_all)} selected by all 4, {len(all_selected)} total union)")


📄 CSV saved to: embedded_selected_features.csv
   Columns: feature | selected_by_Lasso | selected_by_ElasticNet | selected_by_RF | selected_by_XGBoost | selected_by_ALL | n_methods | lasso_coef | elasticnet_coef | rf_importance | xgb_importance
   Rows: 139 features (2 selected by all 4, 139 total union)


# visuals !

In [39]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib_venn import venn3, venn2

from collections import Counter

In [40]:
# 0. Load CSVs
# ══════════════════════════════════════════════════════════════════════════════
filter_df  = pd.read_csv('selected_features.csv')
wrapper_df = pd.read_csv('wrapper_selected_features.csv')
embedded_df= pd.read_csv('embedded_selected_features.csv')

In [41]:
# ── Extract feature sets per individual method ────────────────────────────────
# Filter
anova_feats = set(filter_df[filter_df['selected_by_ANOVA']]['feature'])
mi_feats    = set(filter_df[filter_df['selected_by_MI']]['feature'])
 
# Wrapper
rfe_svm_feats = set(wrapper_df[wrapper_df['selected_by_RFE_SVM']]['feature'])
rfe_lr_feats  = set(wrapper_df[wrapper_df['selected_by_RFE_LogReg']]['feature'])
boruta_feats  = set(wrapper_df[wrapper_df['selected_by_Boruta']]['feature'])
 
# Embedded
lasso_feats = set(embedded_df[embedded_df['selected_by_Lasso']]['feature'])
enet_feats  = set(embedded_df[embedded_df['selected_by_ElasticNet']]['feature'])
rf_feats    = set(embedded_df[embedded_df['selected_by_RF']]['feature'])
xgb_feats   = set(embedded_df[embedded_df['selected_by_XGBoost']]['feature'])
 
# ── Category-level sets (union per type) ─────────────────────────────────────
filter_all   = anova_feats   | mi_feats
wrapper_all  = rfe_svm_feats | rfe_lr_feats | boruta_feats
embedded_all = lasso_feats   | enet_feats   | rf_feats | xgb_feats
 
# ── Global feature frequency (across all 9 individual methods) ────────────────
all_method_sets = {
    'ANOVA':      anova_feats,
    'Mutual Info':mi_feats,
    'RFE-SVM':    rfe_svm_feats,
    'RFE-LogReg': rfe_lr_feats,
    'Boruta':     boruta_feats,
    'Lasso':      lasso_feats,
    'ElasticNet': enet_feats,
    'RF-Embed':   rf_feats,
    'XGBoost':    xgb_feats,
}
 
freq_counter = Counter()
for feats in all_method_sets.values():
    freq_counter.update(feats)
 
freq_df = pd.DataFrame(freq_counter.items(), columns=['feature', 'count'])
freq_df = freq_df.sort_values('count', ascending=False).reset_index(drop=True)

In [42]:
# Palette & style
# ══════════════════════════════════════════════════════════════════════════════
FILTER_CLR   = '#4CC9F0'
WRAPPER_CLR  = '#F72585'
EMBEDDED_CLR = '#7B2FBE'
COMMON_CLR   = '#F8961E'
BG           = '#0D0D0D'
PANEL        = '#161616'
TEXT         = '#E8E8E8'
SUBTEXT      = '#888888'
GRID         = '#2A2A2A'
 
METHOD_COLORS = {
    'ANOVA':       '#4CC9F0',
    'Mutual Info': '#3A86FF',
    'RFE-SVM':     '#F72585',
    'RFE-LogReg':  '#FF006E',
    'Boruta':      '#FF4D6D',
    'Lasso':       '#9B5DE5',
    'ElasticNet':  '#7B2FBE',
    'RF-Embed':    '#5C0099',
    'XGBoost':     '#C77DFF',
}
 
plt.rcParams.update({
    'figure.facecolor':  BG,
    'axes.facecolor':    PANEL,
    'axes.edgecolor':    GRID,
    'axes.labelcolor':   TEXT,
    'xtick.color':       SUBTEXT,
    'ytick.color':       SUBTEXT,
    'text.color':        TEXT,
    'grid.color':        GRID,
    'grid.linewidth':    0.5,
    'font.family':       'monospace',
})

In [43]:
# FIG 1 — Feature frequency heatmap: which features appear across how many methods
# ══════════════════════════════════════════════════════════════════════════════
fig1, axes = plt.subplots(1, 2, figsize=(18, 7),
                           gridspec_kw={'width_ratios': [2, 1]})
fig1.patch.set_facecolor(BG)
fig1.suptitle('FEATURE FREQUENCY ACROSS ALL 9 SELECTION METHODS',
              fontsize=14, fontweight='bold', color=TEXT, y=1.01)
 
# Left: bar chart of count distribution
ax = axes[0]
count_dist = freq_df['count'].value_counts().sort_index()
bars = ax.bar(count_dist.index, count_dist.values,
              color=[plt.cm.plasma(c / 9) for c in count_dist.index],
              edgecolor=BG, linewidth=1.5, width=0.7)
ax.set_xlabel('Number of methods that selected the feature', fontsize=11)
ax.set_ylabel('Number of features', fontsize=11)
ax.set_title('How many methods agree on each feature?', color=SUBTEXT, fontsize=10)
ax.set_xticks(range(1, 10))
ax.grid(axis='y', alpha=0.3)
ax.set_facecolor(PANEL)
for bar, val in zip(bars, count_dist.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=9, color=TEXT)
 
# Right: top 20 most selected features
ax2 = axes[1]
top20 = freq_df.head(20)
colors_top = [plt.cm.plasma(c / 9) for c in top20['count']]
bars2 = ax2.barh(range(len(top20)), top20['count'], color=colors_top,
                  edgecolor=BG, linewidth=1)
ax2.set_yticks(range(len(top20)))
ax2.set_yticklabels(top20['feature'], fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel('Times selected', fontsize=10)
ax2.set_title('Top 20 most selected features', color=SUBTEXT, fontsize=10)
ax2.set_xlim(0, 9.5)
ax2.set_xticks(range(0, 10))
ax2.grid(axis='x', alpha=0.3)
ax2.set_facecolor(PANEL)
for bar, val in zip(bars2, top20['count']):
    ax2.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=8, color=TEXT)
 
# Colorbar legend
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(1, 9))
sm.set_array([])
cbar = fig1.colorbar(sm, ax=axes, orientation='horizontal',
                      fraction=0.03, pad=0.12, aspect=40)
cbar.set_label('Number of methods selecting the feature', color=TEXT, fontsize=10)
cbar.ax.xaxis.set_tick_params(color=SUBTEXT)
plt.setp(cbar.ax.xaxis.get_ticklabels(), color=SUBTEXT)
 
plt.tight_layout()
plt.savefig('fig1_feature_frequency.png', dpi=150, bbox_inches='tight',
            facecolor=BG)
plt.close()
print("✓ fig1_feature_frequency.png")

✓ fig1_feature_frequency.png


In [44]:
# FIG 2 — Per-method selection overlap matrix (heatmap)
# ══════════════════════════════════════════════════════════════════════════════
method_names = list(all_method_sets.keys())
n = len(method_names)
overlap_matrix = np.zeros((n, n), dtype=int)
 
for i, (n1, s1) in enumerate(all_method_sets.items()):
    for j, (n2, s2) in enumerate(all_method_sets.items()):
        overlap_matrix[i, j] = len(s1 & s2)
 
fig2, ax = plt.subplots(figsize=(11, 9))
fig2.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)
 
im = ax.imshow(overlap_matrix, cmap='magma', aspect='auto')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(method_names, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(method_names, fontsize=10)
 
for i in range(n):
    for j in range(n):
        val = overlap_matrix[i, j]
        brightness = val / overlap_matrix.max()
        text_color = 'black' if brightness > 0.6 else TEXT
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=9, color=text_color, fontweight='bold')
 
cbar2 = fig2.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar2.set_label('Number of common features', color=TEXT)
cbar2.ax.yaxis.set_tick_params(color=SUBTEXT)
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color=SUBTEXT)
 
ax.set_title('PAIRWISE FEATURE OVERLAP BETWEEN ALL 9 METHODS',
             fontsize=13, fontweight='bold', color=TEXT, pad=15)
 
# Draw type-group separators
for pos in [1.5, 4.5]:
    ax.axhline(pos, color=BG, linewidth=3)
    ax.axvline(pos, color=BG, linewidth=3)
 
# Group labels
for label, x in [('FILTER', 0.5), ('WRAPPER', 3), ('EMBEDDED', 7)]:
    ax.text(x, -1.2, label, ha='center', fontsize=9,
            color=SUBTEXT, fontweight='bold')
 
plt.tight_layout()
plt.savefig('fig2_overlap_matrix.png', dpi=150, bbox_inches='tight',
            facecolor=BG)
plt.close()
print("✓ fig2_overlap_matrix.png")

✓ fig2_overlap_matrix.png


In [45]:
# FIG 3 — Venn diagram: Filter vs Wrapper vs Embedded (category level)
# ══════════════════════════════════════════════════════════════════════════════
fig3, ax = plt.subplots(figsize=(10, 8))
fig3.patch.set_facecolor(BG)
ax.set_facecolor(BG)
 
v = venn3(
    [filter_all, wrapper_all, embedded_all],
    set_labels=('Filter', 'Wrapper', 'Embedded'),
    set_colors=(FILTER_CLR, WRAPPER_CLR, EMBEDDED_CLR),
    alpha=0.55,
    ax=ax
)
 
# Style labels
for label in v.set_labels:
    if label:
        label.set_fontsize(13)
        label.set_fontweight('bold')
        label.set_color(TEXT)
 
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)
        label.set_color(TEXT)
        label.set_fontweight('bold')
 
ax.set_title('FEATURE OVERLAP ACROSS SELECTION TYPES\n(union of all methods per type)',
             fontsize=13, fontweight='bold', color=TEXT, pad=20)
 
# Annotation
common_3 = filter_all & wrapper_all & embedded_all
ax.annotate(f'{len(common_3)} features\nselected by\nall 3 types',
            xy=(0, 0), xytext=(0.72, 0.15),
            xycoords='axes fraction',
            fontsize=10, color=COMMON_CLR, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=PANEL,
                      edgecolor=COMMON_CLR, alpha=0.9))
 
plt.tight_layout()
plt.savefig('fig3_venn_three_types.png', dpi=150, bbox_inches='tight',
            facecolor=BG)
plt.close()
print("✓ fig3_venn_three_types.png")

✓ fig3_venn_three_types.png


In [46]:
# FIG 4 — Common features across all 3 types: detailed breakdown
# ══════════════════════════════════════════════════════════════════════════════
common_all_3 = sorted(filter_all & wrapper_all & embedded_all)
common_filter_wrapper  = sorted((filter_all & wrapper_all)  - embedded_all)
common_filter_embedded = sorted((filter_all & embedded_all) - wrapper_all)
common_wrapper_embedded= sorted((wrapper_all & embedded_all)- filter_all)
 
fig4, axes4 = plt.subplots(2, 2, figsize=(16, 12))
fig4.patch.set_facecolor(BG)
fig4.suptitle('COMMON FEATURES — DETAILED BREAKDOWN',
              fontsize=15, fontweight='bold', color=TEXT, y=1.01)
 
panels = [
    (axes4[0,0], common_all_3,          'All 3 Types (Filter ∩ Wrapper ∩ Embedded)', COMMON_CLR),
    (axes4[0,1], common_filter_wrapper,  'Filter ∩ Wrapper only',                    '#90E0EF'),
    (axes4[1,0], common_filter_embedded, 'Filter ∩ Embedded only',                   '#C77DFF'),
    (axes4[1,1], common_wrapper_embedded,'Wrapper ∩ Embedded only',                  '#FF6B9D'),
]
 
for ax, feats, title, color in panels:
    ax.set_facecolor(PANEL)
    ax.set_title(f'{title}\n({len(feats)} features)',
                 color=color, fontsize=11, fontweight='bold', pad=10)
 
    if len(feats) == 0:
        ax.text(0.5, 0.5, 'No common features', ha='center', va='center',
                fontsize=12, color=SUBTEXT, transform=ax.transAxes)
        ax.axis('off')
        continue
 
    # Show as a styled table-like list
    cols = 3
    rows = int(np.ceil(len(feats) / cols))
    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.axis('off')
 
    for idx, feat in enumerate(feats):
        col = idx % cols
        row = rows - 1 - (idx // cols)
        # Count how many individual methods selected it
        cnt = freq_counter[feat]
        alpha = 0.4 + 0.6 * (cnt / 9)
        rect = FancyBboxPatch((col + 0.03, row + 0.08), 0.9, 0.75,
                               boxstyle='round,pad=0.05',
                               facecolor=color, alpha=alpha,
                               edgecolor='none')
        ax.add_patch(rect)
        ax.text(col + 0.48, row + 0.46, feat,
                ha='center', va='center', fontsize=7.5,
                color='white', fontweight='bold')
        ax.text(col + 0.48, row + 0.15, f'{cnt}/9 methods',
                ha='center', va='center', fontsize=6,
                color='white', alpha=0.8)
 
plt.tight_layout()
plt.savefig('fig4_common_features.png', dpi=150, bbox_inches='tight',
            facecolor=BG)
plt.close()
print("✓ fig4_common_features.png")

✓ fig4_common_features.png


In [47]:
# FIG 5 — Per-method bar chart: how many features each method selected
#          and overlap with the other methods
# ══════════════════════════════════════════════════════════════════════════════
fig5, ax = plt.subplots(figsize=(14, 6))
fig5.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)
 
method_list  = list(all_method_sets.keys())
method_sizes = [len(all_method_sets[m]) for m in method_list]
bar_colors   = [METHOD_COLORS[m] for m in method_list]
 
# Portion that overlaps with common_all_3
common3_overlap = [len(all_method_sets[m] & set(common_all_3)) for m in method_list]
 
x = np.arange(len(method_list))
width = 0.6
 
bars_total  = ax.bar(x, method_sizes,   width, color=bar_colors, alpha=0.4,
                      edgecolor=bar_colors, linewidth=1.5, label='Total selected')
bars_common = ax.bar(x, common3_overlap, width, color=bar_colors, alpha=1.0,
                      edgecolor='white', linewidth=0.5,
                      label='Overlap with all-3-types common')
 
ax.set_xticks(x)
ax.set_xticklabels(method_list, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Number of features', fontsize=11)
ax.set_title('FEATURES SELECTED PER METHOD vs OVERLAP WITH CROSS-TYPE CONSENSUS',
             fontsize=12, fontweight='bold', color=TEXT, pad=12)
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9, facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT)
 
# Type separators
for pos, label, color in [(1.5, 'FILTER', FILTER_CLR),
                           (4.5, 'WRAPPER', WRAPPER_CLR),
                           (8.5, 'EMBEDDED', EMBEDDED_CLR)]:
    if pos < len(method_list) - 0.5:
        ax.axvline(pos, color=GRID, linewidth=1.5, linestyle='--', alpha=0.6)
 
type_labels = [('FILTER', 0.5), ('WRAPPER', 3), ('EMBEDDED', 6.5)]
for label, xpos in type_labels:
    ax.text(xpos, ax.get_ylim()[1] * 0.97, label,
            ha='center', fontsize=9, color=SUBTEXT, fontweight='bold')
 
# Value labels
for bar, val in zip(bars_total, method_sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=9, color=TEXT)
 
plt.tight_layout()
plt.savefig('fig5_per_method_summary.png', dpi=150, bbox_inches='tight',
            facecolor=BG)
plt.close()
print("✓ fig5_per_method_summary.png")

✓ fig5_per_method_summary.png


In [48]:
# Final summary print
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("CROSS-TYPE FEATURE CONSENSUS SUMMARY")
print("="*60)
print(f"  Filter   methods union  : {len(filter_all)} unique features")
print(f"  Wrapper  methods union  : {len(wrapper_all)} unique features")
print(f"  Embedded methods union  : {len(embedded_all)} unique features")
print(f"  Grand union (all 9)     : {len(filter_all|wrapper_all|embedded_all)} unique features")
print(f"  Filter ∩ Wrapper        : {len(filter_all & wrapper_all)}")
print(f"  Filter ∩ Embedded       : {len(filter_all & embedded_all)}")
print(f"  Wrapper ∩ Embedded      : {len(wrapper_all & embedded_all)}")
print(f"  Filter ∩ Wrapper ∩ Emb  : {len(common_all_3)} ← most trusted features")
print("="*60)
 
if common_all_3:
    print(f"\n🏆 Features selected by ALL 3 types ({len(common_all_3)} total):")
    for i, f in enumerate(common_all_3, 1):
        print(f"   {i:>3}. {f}  [{freq_counter[f]}/9 individual methods]")
 
# Save common features to CSV
pd.DataFrame({
    'feature':          common_all_3,
    'n_individual_methods': [freq_counter[f] for f in common_all_3],
    'in_filter':        [f in filter_all   for f in common_all_3],
    'in_wrapper':       [f in wrapper_all  for f in common_all_3],
    'in_embedded':      [f in embedded_all for f in common_all_3],
}).to_csv('consensus_features.csv', index=False)
print("\n📄 consensus_features.csv saved — features selected by all 3 types")
print("\n📊 Figures saved:")
print("   fig1_feature_frequency.png  — frequency distribution + top 20")
print("   fig2_overlap_matrix.png     — pairwise overlap heatmap (9×9)")
print("   fig3_venn_three_types.png   — Venn diagram Filter/Wrapper/Embedded")
print("   fig4_common_features.png    — all common feature breakdowns")
print("   fig5_per_method_summary.png — per-method totals vs consensus overlap")


CROSS-TYPE FEATURE CONSENSUS SUMMARY
  Filter   methods union  : 80 unique features
  Wrapper  methods union  : 161 unique features
  Embedded methods union  : 139 unique features
  Grand union (all 9)     : 247 unique features
  Filter ∩ Wrapper        : 60
  Filter ∩ Embedded       : 42
  Wrapper ∩ Embedded      : 70
  Filter ∩ Wrapper ∩ Emb  : 39 ← most trusted features

🏆 Features selected by ALL 3 types (39 total):
     1. feature_1152  [4/9 individual methods]
     2. feature_1324  [7/9 individual methods]
     3. feature_1345  [6/9 individual methods]
     4. feature_137  [3/9 individual methods]
     5. feature_1413  [3/9 individual methods]
     6. feature_1422  [6/9 individual methods]
     7. feature_1493  [3/9 individual methods]
     8. feature_1581  [6/9 individual methods]
     9. feature_1634  [5/9 individual methods]
    10. feature_1670  [5/9 individual methods]
    11. feature_1770  [6/9 individual methods]
    12. feature_1771  [8/9 individual methods]
    13. feat

## methods 

# SBE + ANN

In [49]:
import copy, time, warnings
warnings.filterwarnings('ignore')
 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, f1_score,
                              roc_auc_score, classification_report,
                              confusion_matrix)
from imblearn.over_sampling import SMOTE
 
# ── 0. Load consensus features & build index map ─────────────────────────────
consensus_df  = pd.read_csv('consensus_features.csv')          # 39 rows
consensus_feats = list(consensus_df
                       .sort_values('n_individual_methods', ascending=False)
                       ['feature'])                            # sorted by trust
 
def feat_to_idx(feat_list):
    """'feature_N'  →  integer column index N  (0-based)."""
    return np.array([int(f.split('_')[1]) for f in feat_list])
 
# ── 1. Shared CV / SMOTE setup ───────────────────────────────────────────────
CV_FOLDS = 5
SMOTE_K  = 3                   # conservative for small minority class
cv    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
smote = SMOTE(random_state=42, k_neighbors=SMOTE_K)
 
# KNN – best classifier from the benchmark
knn_base = KNeighborsClassifier(n_neighbors=5)
 
def cv_score(feat_list, clf=None):
    """
    Run 5-fold CV on a feature subset.
    SMOTE + scaling applied ONLY inside each training fold.
    Returns mean balanced accuracy.
    """
    if clf is None:
        clf = KNeighborsClassifier(n_neighbors=5)
    idx   = feat_to_idx(feat_list)
    X_sub = X[:, idx]
    scores = []
    for tr, te in cv.split(X_sub, y):
        X_tr, y_tr = X_sub[tr], y[tr]
        X_te, y_te = X_sub[te], y[te]
        # SMOTE on train only
        X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
        # Scale on train only
        sc = StandardScaler()
        X_tr_res = sc.fit_transform(X_tr_res)
        X_te_sc  = sc.transform(X_te)
        c = clone(clf)
        c.fit(X_tr_res, y_tr_res)
        pred = c.predict(X_te_sc)
        scores.append(balanced_accuracy_score(y_te, pred))
    return np.mean(scores), np.std(scores)
 
# ── 2. SBE on 39 consensus features ──────────────────────────────────────────
print("=" * 65)
print("STEP 1 — Sequential Backward Elimination (SBE) on 39 features")
print("=" * 65)
 
remaining   = list(consensus_feats)          # start: all 39
sbe_log     = []
MIN_FEATURES = 5
 
base_score, base_std = cv_score(remaining)
sbe_log.append({'N': len(remaining), 'Removed': None,
                'BalAcc': base_score, 'Std': base_std})
best_score  = base_score
best_subset = remaining.copy()
 
print(f"  Start : {len(remaining):>2} features | "
      f"BalAcc = {base_score:.4f} ± {base_std:.4f}")
 
while len(remaining) > MIN_FEATURES:
    candidate_scores = {}
    for feat in remaining:
        trial = [f for f in remaining if f != feat]
        s, _  = cv_score(trial)
        candidate_scores[feat] = s
 
    # remove the feature whose absence hurts the LEAST (or helps the most)
    drop_feat  = max(candidate_scores, key=candidate_scores.get)
    drop_score = candidate_scores[drop_feat]
    remaining.remove(drop_feat)
 
    _, drop_std = cv_score(remaining)
    sbe_log.append({'N': len(remaining), 'Removed': drop_feat,
                    'BalAcc': drop_score, 'Std': drop_std})
 
    print(f"  {len(remaining):>2} feats | "
          f"−{drop_feat:<18} | BalAcc = {drop_score:.4f} ± {drop_std:.4f}")
 
    if drop_score >= best_score:
        best_score  = drop_score
        best_subset = remaining.copy()
 
df_sbe = pd.DataFrame(sbe_log)
df_sbe.to_csv('pipeline1_sbe_path.csv', index=False)
 
print(f"\n  ✔  Best SBE subset : {len(best_subset)} features  "
      f"BalAcc = {best_score:.4f}")
print(f"  Features kept : {best_subset}")
 
 
# ── 3. Attention-gated MLP ────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 2 — Attention-gated ANN on SBE-selected subset")
print("=" * 65)
 
N_FEATS = len(best_subset)
 
class AttentionMLP(nn.Module):
    """
    Soft-attention gate  →  2-layer MLP.
    gate(x) = sigmoid(W·x + b)   ∈ (0,1)  per feature
    gated_x  = gate(x) * x
    The gate weights are the learned feature importance.
    """
    def __init__(self, n_in, hidden=32, dropout=0.4):
        super().__init__()
        # gate: same dim as input, learned per-feature weight
        self.gate = nn.Sequential(
            nn.Linear(n_in, n_in),
            nn.Sigmoid()
        )
        self.mlp = nn.Sequential(
            nn.Linear(n_in,     hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden,   hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1)        # single logit → BCEWithLogitsLoss
        )
 
    def forward(self, x):
        return self.mlp(self.gate(x) * x).squeeze(1)
 
    def gate_weights(self, x):
        """Return mean gate activation across samples (feature importance)."""
        with torch.no_grad():
            return self.gate(x).mean(0).cpu().numpy()
 
 
def train_attention_mlp(X_tr, y_tr, n_in,
                        hidden=32, dropout=0.4,
                        epochs=300, lr=5e-4,
                        batch_size=16, patience=30):
    """
    Train AttentionMLP with early stopping.
    pos_weight handles class imbalance without extra SMOTE inside NN training
    (SMOTE already applied before this call).
    """
    pos_w  = torch.tensor([(1 - y_tr).sum() / max(y_tr.sum(), 1)],
                           dtype=torch.float32)
    model  = AttentionMLP(n_in, hidden, dropout)
    opt    = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit   = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
 
    Xt = torch.FloatTensor(X_tr.astype(np.float32))
    yt = torch.FloatTensor(y_tr.astype(np.float32))
    loader = DataLoader(TensorDataset(Xt, yt),
                        batch_size=batch_size, shuffle=True)
 
    best_loss, wait, best_state = float('inf'), 0, None
    model.train()
    for _ in range(epochs):
        epoch_loss = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        sched.step()
        avg = epoch_loss / len(loader)
        if avg < best_loss - 1e-4:
            best_loss  = avg
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
 
    model.load_state_dict(best_state)
    model.eval()
    return model
 
 
def predict_mlp(model, X_te):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X_te.astype(np.float32)))
        probs  = torch.sigmoid(logits).numpy()
    return probs, (probs >= 0.5).astype(int)
 
 
# ── 4. Nested CV for final evaluation ────────────────────────────────────────
idx_best  = feat_to_idx(best_subset)
X_best    = X[:, idx_best]
 
fold_ba, fold_f1, fold_auc = [], [], []
all_gate_weights = []          # collect per-fold importance
 
for fold_i, (tr, te) in enumerate(cv.split(X_best, y)):
    X_tr, y_tr = X_best[tr], y[tr]
    X_te, y_te = X_best[te], y[te]
 
    # SMOTE only on train
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
 
    # Scale: fit on train, apply to test
    sc       = StandardScaler()
    X_tr_sc  = sc.fit_transform(X_tr_res)
    X_te_sc  = sc.transform(X_te)
 
    # Train ANN
    model = train_attention_mlp(X_tr_sc, y_tr_res, n_in=N_FEATS)
 
    # Predict
    probs, preds = predict_mlp(model, X_te_sc)
 
    fold_ba.append(balanced_accuracy_score(y_te, preds))
    fold_f1.append(f1_score(y_te, preds, zero_division=0))
    if len(np.unique(y_te)) > 1:
        fold_auc.append(roc_auc_score(y_te, probs))
 
    # Gate weights on the test split (unseen data)
    all_gate_weights.append(
        model.gate_weights(torch.FloatTensor(X_te_sc.astype(np.float32)))
    )
    print(f"  Fold {fold_i+1} | BalAcc={fold_ba[-1]:.4f}  "
          f"F1={fold_f1[-1]:.4f}  AUC={fold_auc[-1]:.4f}")
 
# ── 5. Results ────────────────────────────────────────────────────────────────
mean_ba  = np.mean(fold_ba);   std_ba  = np.std(fold_ba)
mean_f1  = np.mean(fold_f1);   std_f1  = np.std(fold_f1)
mean_auc = np.mean(fold_auc);  std_auc = np.std(fold_auc)
 
print(f"\n{'='*65}")
print(f"  PIPELINE 1 FINAL RESULTS  ({N_FEATS} features, SBE → ANN)")
print(f"{'='*65}")
print(f"  Balanced Accuracy : {mean_ba:.4f} ± {std_ba:.4f}")
print(f"  F1 Score          : {mean_f1:.4f} ± {std_f1:.4f}")
print(f"  ROC-AUC           : {mean_auc:.4f} ± {std_auc:.4f}")
 
# ── 6. Stable feature importance (averaged gate weights across folds) ─────────
mean_gate = np.mean(all_gate_weights, axis=0)
importance_df = pd.DataFrame({
    'feature':          best_subset,
    'gate_weight_mean': mean_gate,
    'n_methods':        [int(consensus_df.set_index('feature')
                             .loc[f, 'n_individual_methods'])
                         for f in best_subset]
}).sort_values('gate_weight_mean', ascending=False).reset_index(drop=True)
 
print(f"\n  Top-10 features by attention gate weight:")
print(importance_df.head(10).to_string(index=False))
 
importance_df.to_csv('pipeline1_feature_importance.csv', index=False)
print("\n  📄 pipeline1_feature_importance.csv saved")
print("  📄 pipeline1_sbe_path.csv saved")

STEP 1 — Sequential Backward Elimination (SBE) on 39 features
  Start : 39 features | BalAcc = 0.8875 ± 0.0919
  38 feats | −feature_764        | BalAcc = 0.8875 ± 0.0919
  37 feats | −feature_492        | BalAcc = 0.8875 ± 0.0919
  36 feats | −feature_1771       | BalAcc = 0.8875 ± 0.0919
  35 feats | −feature_248        | BalAcc = 0.8875 ± 0.0919
  34 feats | −feature_1324       | BalAcc = 0.8875 ± 0.0919
  33 feats | −feature_376        | BalAcc = 0.8875 ± 0.0919
  32 feats | −feature_1422       | BalAcc = 0.8875 ± 0.0919
  31 feats | −feature_1345       | BalAcc = 0.8875 ± 0.0919
  30 feats | −feature_285        | BalAcc = 0.8875 ± 0.0919
  29 feats | −feature_1770       | BalAcc = 0.8875 ± 0.0919
  28 feats | −feature_1634       | BalAcc = 0.8875 ± 0.0919
  27 feats | −feature_1581       | BalAcc = 0.8875 ± 0.0919
  26 feats | −feature_624        | BalAcc = 0.8875 ± 0.0919
  25 feats | −feature_1670       | BalAcc = 0.8875 ± 0.0919
  24 feats | −feature_1152       | BalAcc = 0.887

# GA

In [54]:
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter
 
# ── 0. Setup ──────────────────────────────────────────────────────────────────
consensus_df   = pd.read_csv('consensus_features.csv')
consensus_feats = list(consensus_df
                       .sort_values('n_individual_methods', ascending=False)
                       ['feature'])
N_POOL = len(consensus_feats)          # 39
 
def feat_to_idx(feat_list):
    return np.array([int(f.split('_')[1]) for f in feat_list])
 
pool_idx = feat_to_idx(consensus_feats)   # column indices in X
 
CV_FOLDS = 5
SMOTE_K  = 3
cv    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
smote = SMOTE(random_state=42, k_neighbors=SMOTE_K)
knn   = KNeighborsClassifier(n_neighbors=5)
 
# ── 1. Fitness function ───────────────────────────────────────────────────────
TARGET_SIZE = 15          # ideal subset size (penalty kicks in above this)
LAMBDA      = 0.004       # penalty per extra feature
 
def fitness(chromosome):
    """
    Evaluate a binary chromosome (length 39) using 5-fold CV.
    Returns penalised balanced accuracy.
    """
    sel_idx = pool_idx[chromosome.astype(bool)]
    n_sel   = len(sel_idx)
    if n_sel < 3:
        return 0.0          # degenerate solution
 
    X_sub  = X[:, sel_idx]
    scores = []
    for tr, te in cv.split(X_sub, y):
        X_tr, y_tr = X_sub[tr], y[tr]
        X_te, y_te = X_sub[te], y[te]
        X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
        sc = StandardScaler()
        X_tr_sc = sc.fit_transform(X_tr_res)
        X_te_sc = sc.transform(X_te)
        c = clone(knn)
        c.fit(X_tr_sc, y_tr_res)
        scores.append(balanced_accuracy_score(y_te, c.predict(X_te_sc)))
 
    penalty = max(0, n_sel - TARGET_SIZE) * LAMBDA
    return np.mean(scores) - penalty
 
# ── 2. GA operators ───────────────────────────────────────────────────────────
POP_SIZE    = 20
N_GEN       = 30
MUTATION_R  = 0.05     # probability of flipping each bit
CROSSOVER_R = 0.75     # probability of doing crossover vs cloning
TOURNAMENT_K= 3        # tournament selection size
ELITISM     = 2        # number of best individuals kept each generation
CONVERGE_GEN= 8        # stop early if best fitness unchanged for this many gens
 
def init_population(pop_size, n_genes, target_k):
    """Initialise with chromosomes that have ~target_k bits set."""
    pop = np.zeros((pop_size, n_genes), dtype=int)
    for i in range(pop_size):
        chosen = np.random.choice(n_genes, target_k, replace=False)
        pop[i, chosen] = 1
    return pop
 
def tournament_select(pop, fit_scores, k=TOURNAMENT_K):
    """Pick k random individuals and return the fittest."""
    idx = np.random.choice(len(pop), k, replace=False)
    return pop[idx[np.argmax(fit_scores[idx])]].copy()
 
def crossover(p1, p2):
    """Single-point crossover."""
    if np.random.rand() > CROSSOVER_R:
        return p1.copy(), p2.copy()
    pt = np.random.randint(1, len(p1))
    c1 = np.concatenate([p1[:pt], p2[pt:]])
    c2 = np.concatenate([p2[:pt], p1[pt:]])
    return c1, c2
 
def mutate(chrom, rate=MUTATION_R):
    """Bit-flip mutation."""
    c = chrom.copy()
    flip = np.random.rand(len(c)) < rate
    c[flip] = 1 - c[flip]
    return c
 
# ── 3. Multi-run GA ───────────────────────────────────────────────────────────
N_RUNS = 10           # independent runs to measure stability
 
all_best_chroms  = []    # best chromosome per run
all_best_scores  = []    # best fitness per run
all_histories    = []    # convergence curve per run
 
print("=" * 65)
print(f"PIPELINE 2 — Genetic Algorithm  ({N_RUNS} independent runs)")
print("=" * 65)
 
for run in range(N_RUNS):
    np.random.seed(run * 7 + 13)        # different seed each run
    pop = init_population(POP_SIZE, N_POOL, TARGET_SIZE)
    run_best_score = -1
    run_best_chrom = None
    history        = []
    no_improve     = 0
 
    for gen in range(N_GEN):
        fit = np.array([fitness(c) for c in pop])
 
        gen_best_idx   = np.argmax(fit)
        gen_best_score = fit[gen_best_idx]
        history.append(gen_best_score)
 
        if gen_best_score > run_best_score + 1e-5:
            run_best_score = gen_best_score
            run_best_chrom = pop[gen_best_idx].copy()
            no_improve     = 0
        else:
            no_improve += 1
            if no_improve >= CONVERGE_GEN:
                print(f"  Run {run+1:>2} | Gen {gen+1:>2} | "
                      f"converged → best={run_best_score:.4f}  "
                      f"feats={run_best_chrom.sum()}")
                break
 
        # Build next generation
        elite_idx = np.argsort(fit)[-ELITISM:]
        new_pop   = [pop[i].copy() for i in elite_idx]
 
        while len(new_pop) < POP_SIZE:
            p1 = tournament_select(pop, fit)
            p2 = tournament_select(pop, fit)
            c1, c2 = crossover(p1, p2)
            new_pop.append(mutate(c1))
            if len(new_pop) < POP_SIZE:
                new_pop.append(mutate(c2))
 
        pop = np.array(new_pop)
 
    else:
        print(f"  Run {run+1:>2} | Gen {N_GEN:>2} | "
              f"best={run_best_score:.4f}  feats={run_best_chrom.sum()}")
 
    all_best_chroms.append(run_best_chrom)
    all_best_scores.append(run_best_score)
    all_histories.append(history)
 
# ── 4. Stability analysis — feature frequency across runs ─────────────────────
freq_arr = np.array(all_best_chroms).mean(axis=0)   # fraction of runs selecting each feat
 
freq_df = pd.DataFrame({
    'feature':    consensus_feats,
    'frequency':  freq_arr,                          # 0.0 – 1.0
    'n_methods':  list(consensus_df
                       .sort_values('n_individual_methods', ascending=False)
                       ['n_individual_methods'])
}).sort_values('frequency', ascending=False).reset_index(drop=True)
 
print(f"\n  Feature stability across {N_RUNS} GA runs:")
print(freq_df.to_string(index=False))
 
# Stable core: features appearing in ≥ 60 % of runs
STABILITY_THRESH = 0.60
stable_feats = list(freq_df[freq_df['frequency'] >= STABILITY_THRESH]['feature'])
print(f"\n  Stable core (≥{int(STABILITY_THRESH*100)}% of runs): "
      f"{len(stable_feats)} features")
print(f"  {stable_feats}")
 
# ── 5. Final CV evaluation of the stable core ─────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score
 
stable_idx = feat_to_idx(stable_feats)
X_stable   = X[:, stable_idx]
 
ba_list, f1_list, auc_list = [], [], []
for tr, te in cv.split(X_stable, y):
    X_tr, y_tr = X_stable[tr], y[tr]
    X_te, y_te = X_stable[te], y[te]
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr_res)
    X_te_sc = sc.transform(X_te)
    c = clone(knn)
    c.fit(X_tr_sc, y_tr_res)
    prob = c.predict_proba(X_te_sc)[:, 1]
    pred = (prob >= 0.5).astype(int)
    ba_list.append(balanced_accuracy_score(y_te, pred))
    f1_list.append(f1_score(y_te, pred, zero_division=0))
    if len(np.unique(y_te)) > 1:
        auc_list.append(roc_auc_score(y_te, prob))
 
print(f"\n{'='*65}")
print(f"  PIPELINE 2 FINAL RESULTS  ({len(stable_feats)} stable features)")
print(f"{'='*65}")
print(f"  Balanced Accuracy : {np.mean(ba_list):.4f} ± {np.std(ba_list):.4f}")
print(f"  F1 Score          : {np.mean(f1_list):.4f} ± {np.std(f1_list):.4f}")
print(f"  ROC-AUC           : {np.mean(auc_list):.4f} ± {np.std(auc_list):.4f}")
print(f"  GA run scores     : mean={np.mean(all_best_scores):.4f}  "
      f"std={np.std(all_best_scores):.4f}")
 
# Save
freq_df.to_csv('pipeline2_ga_feature_freq.csv', index=False)
pd.DataFrame(all_histories).T.to_csv('pipeline2_ga_convergence.csv', index=False)
pd.DataFrame({'feature': stable_feats}).to_csv('pipeline2_stable_features.csv', index=False)
print("\n  📄 pipeline2_ga_feature_freq.csv     — per-feature stability")
print("  📄 pipeline2_ga_convergence.csv      — convergence per run")
print("  📄 pipeline2_stable_features.csv     — stable core features")
 


PIPELINE 2 — Genetic Algorithm  (10 independent runs)
  Run  1 | Gen 20 | converged → best=0.9375  feats=14
  Run  2 | Gen 19 | converged → best=0.9375  feats=14
  Run  3 | Gen 18 | converged → best=0.9125  feats=15
  Run  4 | Gen 27 | converged → best=0.9375  feats=14
  Run  5 | Gen 28 | converged → best=0.9500  feats=12
  Run  6 | Gen 18 | converged → best=0.9125  feats=15
  Run  7 | Gen 13 | converged → best=0.9250  feats=13
  Run  8 | Gen 11 | converged → best=0.9125  feats=15
  Run  9 | Gen  9 | converged → best=0.9125  feats=15
  Run 10 | Gen  9 | converged → best=0.9125  feats=15

  Feature stability across 10 GA runs:
     feature  frequency  n_methods
 feature_492        0.7          9
 feature_285        0.7          7
feature_1324        0.7          7
 feature_624        0.7          5
 feature_376        0.6          7
feature_1345        0.6          6
feature_1771        0.6          8
 feature_896        0.6          4
 feature_512        0.6          4
 feature_779    

# GA + ANN

In [55]:

import numpy as np
import pandas as pd
import copy, warnings
warnings.filterwarnings('ignore')
 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, f1_score,
                              roc_auc_score, classification_report,
                              confusion_matrix)
from imblearn.over_sampling import SMOTE
 
# ── 0. Load stable core (output of Cell 2) ───────────────────────────────────
try:
    stable_df    = pd.read_csv('pipeline2_stable_features.csv')
    stable_feats = list(stable_df['feature'])
    print(f"  Loaded stable core from pipeline2_stable_features.csv "
          f"({len(stable_feats)} features)")
except FileNotFoundError:
    # Fallback: use all 39 consensus features
    consensus_df = pd.read_csv('consensus_features.csv')
    stable_feats = list(consensus_df
                        .sort_values('n_individual_methods', ascending=False)
                        ['feature'])
    print(f"  pipeline2_stable_features.csv not found — "
          f"using all 39 consensus features as fallback")
 
def feat_to_idx(feat_list):
    return np.array([int(f.split('_')[1]) for f in feat_list])
 
N_FEATS   = len(stable_feats)
stable_idx = feat_to_idx(stable_feats)
X_stable   = X[:, stable_idx]
 
CV_FOLDS = 5
SMOTE_K  = 3
cv    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
smote = SMOTE(random_state=42, k_neighbors=SMOTE_K)
 
print(f"  Input to ANN : {N_FEATS} features")
 
# ── 1. Attention-gated MLP ────────────────────────────────────────────────────
class AttentionMLP(nn.Module):
    """
    Soft-attention gate  ×  input  →  small MLP.
 
    Architecture (scales automatically with n_in):
      gate    : Linear(n_in → n_in)  + Sigmoid
      layer 1 : Linear(n_in → H)     + BN + ReLU + Dropout
      layer 2 : Linear(H    → H//2)  + BN + ReLU + Dropout
      output  : Linear(H//2 → 1)     (raw logit for BCEWithLogitsLoss)
 
    H = max(16, min(64, n_in * 2))   keeps the net small for our dataset size
    """
    def __init__(self, n_in, dropout=0.4):
        super().__init__()
        H = max(16, min(64, n_in * 2))
        self.gate = nn.Sequential(nn.Linear(n_in, n_in), nn.Sigmoid())
        self.mlp  = nn.Sequential(
            nn.Linear(n_in,  H),    nn.BatchNorm1d(H),    nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(H,  H // 2),  nn.BatchNorm1d(H//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(H // 2, 1)
        )
 
    def forward(self, x):
        return self.mlp(self.gate(x) * x).squeeze(1)
 
    def gate_weights(self, x_tensor):
        with torch.no_grad():
            return self.gate(x_tensor).mean(0).cpu().numpy()
 
 
def train_ann(X_tr, y_tr, n_in,
              epochs=350, lr=5e-4, batch_size=16,
              dropout=0.4, patience=35):
    """
    Train AttentionMLP with cosine LR schedule and early stopping.
    X_tr, y_tr already SMOTE-resampled and scaled.
    """
    pos_w  = torch.tensor(
        [(1 - y_tr).sum() / max(y_tr.sum(), 1)], dtype=torch.float32
    )
    model  = AttentionMLP(n_in, dropout)
    opt    = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit   = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
 
    Xt = torch.FloatTensor(X_tr.astype(np.float32))
    yt = torch.FloatTensor(y_tr.astype(np.float32))
    loader = DataLoader(TensorDataset(Xt, yt),
                        batch_size=batch_size, shuffle=True,
                        drop_last=(len(Xt) > batch_size))
 
    best_loss, wait, best_state = float('inf'), 0, None
    model.train()
    for _ in range(epochs):
        e_loss = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            e_loss += loss.item()
        sched.step()
        avg = e_loss / len(loader)
        if avg < best_loss - 1e-4:
            best_loss  = avg
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
 
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    return model
 
 
def predict_ann(model, X_te):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X_te.astype(np.float32)))
        probs  = torch.sigmoid(logits).numpy()
    preds = (probs >= 0.5).astype(int)
    return probs, preds
 
 
# ── 2. Nested 5-fold CV evaluation ───────────────────────────────────────────
print("\n" + "=" * 65)
print("PIPELINE 3 — GA Stable Core → Attention ANN")
print("=" * 65)
 
fold_ba, fold_f1, fold_auc = [], [], []
gate_weights_all = []          # shape: [n_folds, N_FEATS]
all_preds        = np.zeros(len(y), dtype=int)
all_probs        = np.zeros(len(y))
 
for fold_i, (tr, te) in enumerate(cv.split(X_stable, y)):
    X_tr, y_tr = X_stable[tr], y[tr]
    X_te, y_te = X_stable[te], y[te]
 
    # SMOTE only on train
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
 
    # Scale: fit on train only
    sc      = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr_res)
    X_te_sc = sc.transform(X_te)
 
    # Train ANN
    model = train_ann(X_tr_sc, y_tr_res, n_in=N_FEATS)
 
    # Predict
    probs, preds = predict_ann(model, X_te_sc)
 
    # Store OOF predictions (for confusion matrix on full dataset)
    all_preds[te] = preds
    all_probs[te] = probs
 
    fold_ba.append(balanced_accuracy_score(y_te, preds))
    fold_f1.append(f1_score(y_te, preds, zero_division=0))
    if len(np.unique(y_te)) > 1:
        fold_auc.append(roc_auc_score(y_te, probs))
 
    # Gate weights on TEST split (never seen during training)
    gw = model.gate_weights(
        torch.FloatTensor(X_te_sc.astype(np.float32))
    )
    gate_weights_all.append(gw)
 
    print(f"  Fold {fold_i+1} | BalAcc={fold_ba[-1]:.4f}  "
          f"F1={fold_f1[-1]:.4f}  AUC={fold_auc[-1]:.4f}")
 
# ── 3. Final results ──────────────────────────────────────────────────────────
mean_ba  = np.mean(fold_ba);   std_ba  = np.std(fold_ba)
mean_f1  = np.mean(fold_f1);   std_f1  = np.std(fold_f1)
mean_auc = np.mean(fold_auc);  std_auc = np.std(fold_auc)
 
print(f"\n{'='*65}")
print(f"  PIPELINE 3 FINAL RESULTS  ({N_FEATS} features, GA→ANN)")
print(f"{'='*65}")
print(f"  Balanced Accuracy : {mean_ba:.4f} ± {std_ba:.4f}")
print(f"  F1 Score          : {mean_f1:.4f} ± {std_f1:.4f}")
print(f"  ROC-AUC           : {mean_auc:.4f} ± {std_auc:.4f}")
 
print(f"\n  Out-of-fold classification report:")
print(classification_report(y, all_preds,
                             target_names=['negative', 'positive']))
 
print(f"  Confusion matrix (OOF):")
cm = confusion_matrix(y, all_preds)
print(f"    TN={cm[0,0]}  FP={cm[0,1]}")
print(f"    FN={cm[1,0]}  TP={cm[1,1]}")
 
# ── 4. Stable feature importance (mean gate across folds) ─────────────────────
mean_gate = np.mean(gate_weights_all, axis=0)
std_gate  = np.std(gate_weights_all,  axis=0)
 
try:
    consensus_df = pd.read_csv('consensus_features.csv')
    n_methods_map = consensus_df.set_index('feature')['n_individual_methods'].to_dict()
    n_methods_col = [n_methods_map.get(f, 0) for f in stable_feats]
except Exception:
    n_methods_col = [0] * N_FEATS
 
importance_df = pd.DataFrame({
    'feature':          stable_feats,
    'gate_weight_mean': mean_gate,
    'gate_weight_std':  std_gate,
    'n_methods':        n_methods_col,
    'rank':             0
}).sort_values('gate_weight_mean', ascending=False).reset_index(drop=True)
importance_df['rank'] = importance_df.index + 1
 
print(f"\n  Feature importance (mean attention gate, averaged across folds):")
print(importance_df.to_string(index=False))
 
# Save all outputs
importance_df.to_csv('pipeline3_feature_importance.csv', index=False)
 
results_df = pd.DataFrame({
    'Pipeline':         ['GA → Stable Core → Attention ANN'],
    'N_features':       [N_FEATS],
    'Balanced_Acc':     [mean_ba],
    'Balanced_Acc_Std': [std_ba],
    'F1':               [mean_f1],
    'F1_Std':           [std_f1],
    'ROC_AUC':          [mean_auc],
    'ROC_AUC_Std':      [std_auc],
})
results_df.to_csv('pipeline3_results.csv', index=False)
 
print("\n  📄 pipeline3_feature_importance.csv  — feature importance")
print("  📄 pipeline3_results.csv             — final metrics")

  Loaded stable core from pipeline2_stable_features.csv (9 features)
  Input to ANN : 9 features

PIPELINE 3 — GA Stable Core → Attention ANN
  Fold 1 | BalAcc=0.9375  F1=0.9091  AUC=1.0000
  Fold 2 | BalAcc=0.8375  F1=0.8000  AUC=0.9750
  Fold 3 | BalAcc=0.7500  F1=0.6667  AUC=0.8750
  Fold 4 | BalAcc=0.8750  F1=0.8571  AUC=1.0000
  Fold 5 | BalAcc=0.9375  F1=0.8889  AUC=1.0000

  PIPELINE 3 FINAL RESULTS  (9 features, GA→ANN)
  Balanced Accuracy : 0.8675 ± 0.0701
  F1 Score          : 0.8244 ± 0.0870
  ROC-AUC           : 0.9700 ± 0.0485

  Out-of-fold classification report:
              precision    recall  f1-score   support

    negative       0.92      0.88      0.90        40
    positive       0.79      0.86      0.83        22

    accuracy                           0.87        62
   macro avg       0.86      0.87      0.86        62
weighted avg       0.88      0.87      0.87        62

  Confusion matrix (OOF):
    TN=35  FP=5
    FN=3  TP=19

  Feature importance (mean att

## Pca features combined ( not this ) 

In [50]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL — Combined Feature Sets + PCA Extraction
# ══════════════════════════════════════════════════════════════════════════════
#
# This cell extracts feature subsets in the EXACT same way as the rest of the
# notebook (feature_N → column index N, same CSV sources, same variable names)
# and adds two new selection strategies:
#
#   Combined  — union of all 9 method outputs (247 features), treated as one
#               feature set the way the benchmark treated each individual method
#
#   PCA       — dimensionality reduction applied to the grand union (247 features)
#               retaining enough components to explain 95% of variance.
#               PCA is fitted INSIDE each CV fold (no leakage).
#
# Both are then benchmarked with the same classifiers, CV, SMOTE and scoring
# that were used in the Filter / Wrapper / Embedded phases.
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, balanced_accuracy_score, f1_score
from sklearn.base import BaseEstimator, TransformerMixin
import copy, time

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# ── 0. Re-load the same CSV sources the notebook already produced ─────────────
filter_df   = pd.read_csv('selected_features.csv')
wrapper_df  = pd.read_csv('wrapper_selected_features.csv')
embedded_df = pd.read_csv('embedded_selected_features.csv')
consensus_df= pd.read_csv('consensus_features.csv')

# ── 1. Rebuild all 9 individual method feature sets (same as visuals section) ─
anova_feats   = set(filter_df  [filter_df  ['selected_by_ANOVA']]   ['feature'])
mi_feats      = set(filter_df  [filter_df  ['selected_by_MI']]      ['feature'])
rfe_svm_feats = set(wrapper_df [wrapper_df ['selected_by_RFE_SVM']] ['feature'])
rfe_lr_feats  = set(wrapper_df [wrapper_df ['selected_by_RFE_LogReg']]['feature'])
boruta_feats  = set(wrapper_df [wrapper_df ['selected_by_Boruta']]  ['feature'])
lasso_feats   = set(embedded_df[embedded_df['selected_by_Lasso']]   ['feature'])
enet_feats    = set(embedded_df[embedded_df['selected_by_ElasticNet']]['feature'])
rf_feats      = set(embedded_df[embedded_df['selected_by_RF']]      ['feature'])
xgb_feats     = set(embedded_df[embedded_df['selected_by_XGBoost']] ['feature'])

# ── 2. Build the new feature sets ─────────────────────────────────────────────

# Category-level unions  (same as in the visuals section)
filter_all   = anova_feats   | mi_feats
wrapper_all  = rfe_svm_feats | rfe_lr_feats | boruta_feats
embedded_all = lasso_feats   | enet_feats   | rf_feats | xgb_feats

# ① Grand union — every feature selected by at least ONE of the 9 methods
grand_union  = filter_all | wrapper_all | embedded_all   # 247 features

# ② Consensus — features selected by ALL THREE category types (same 39 as before)
consensus_feats = set(consensus_df['feature'])           # 39 features

# Helper: 'feature_N'  →  column index N  (same convention as whole notebook)
def feat_to_idx(feat_set):
    return np.array(sorted([int(f.split('_')[1]) for f in feat_set]))

grand_idx    = feat_to_idx(grand_union)      # shape (247,)
consensus_idx= feat_to_idx(consensus_feats)  # shape (39,)

print("=" * 65)
print("COMBINED FEATURE SETS")
print("=" * 65)
print(f"  Filter   union  : {len(filter_all):>4} features")
print(f"  Wrapper  union  : {len(wrapper_all):>4} features")
print(f"  Embedded union  : {len(embedded_all):>4} features")
print(f"  Grand union     : {len(grand_union):>4} features  ← new 'Combined' set")
print(f"  Consensus (×3)  : {len(consensus_feats):>4} features  (reference)")

# ── 3. Shared setup — identical to the rest of the notebook ───────────────────
K_FEATURES = 50
CV_FOLDS   = 5
SMOTE_K    = 3

cv    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
smote = SMOTE(random_state=42, k_neighbors=SMOTE_K)

scoring = {
    'balanced_acc': make_scorer(balanced_accuracy_score),
    'f1':           make_scorer(f1_score),
    'roc_auc':      'roc_auc',
}

classifiers = {
    'KNN':           KNeighborsClassifier(n_neighbors=5),
    'SVM':           SVC(kernel='rbf', class_weight='balanced',
                         probability=True, random_state=42),
    'Naive Bayes':   GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced',
                                             random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100,
                                             class_weight='balanced',
                                             random_state=42),
    'Logistic Reg':  LogisticRegression(class_weight='balanced',
                                         max_iter=1000, random_state=42),
}

# ── 4. PCA transformer — fits INSIDE each fold (no leakage) ───────────────────
class SafePCA(BaseEstimator, TransformerMixin):
    """
    PCA wrapper that deep-copies itself each fold so it is always
    re-fitted from scratch — same pattern used for RFE/Boruta in the
    wrapper section of this notebook.

    variance_ratio: keep enough components to explain this fraction of
                    variance (computed on the training fold).
    """
    def __init__(self, variance_ratio=0.95):
        self.variance_ratio = variance_ratio

    def fit(self, X, y=None):
        self._pca = PCA(n_components=self.variance_ratio,
                        random_state=42)
        self._pca.fit(X)
        return self

    def transform(self, X):
        return self._pca.transform(X)

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


class SafePCAWrapper(BaseEstimator, TransformerMixin):
    """Deep-copies SafePCA each fold — prevents state leakage across folds."""
    def __init__(self, variance_ratio=0.95):
        self.variance_ratio = variance_ratio

    def fit(self, X, y=None):
        self._fitted = copy.deepcopy(SafePCA(self.variance_ratio))
        self._fitted.fit(X)
        return self

    def transform(self, X):
        return self._fitted.transform(X)


# ── 5. Run benchmark ───────────────────────────────────────────────────────────
#
# Four feature-selection strategies benchmarked:
#
#   "Combined (all)"   — grand union of all 247 features, no further reduction
#   "Combined + PCA"   — grand union → PCA (95% variance) inside each fold
#   "Consensus"        — the 39 features (reference, same as earlier phases)
#   "Consensus + PCA"  — the 39 features → PCA inside each fold
#
# Each paired with all 6 classifiers, 5-fold CV, SMOTE on train only.

results = []

strategies = {
    # name                  : (feature_indices,  selector_step)
    'Combined (all)':        (grand_idx,     None),
    'Combined + PCA':        (grand_idx,     SafePCAWrapper(0.95)),
    'Consensus':             (consensus_idx, None),
    'Consensus + PCA':       (consensus_idx, SafePCAWrapper(0.95)),
}

print("\n" + "=" * 75)
print(f"{'Strategy':<22} | {'Classifier':<16} | "
      f"{'BalAcc':>7}  {'F1':>6}  {'AUC':>6}  {'Time':>6}")
print("-" * 75)

for strat_name, (feat_idx, selector) in strategies.items():
    X_sub = X[:, feat_idx]                 # pre-slice once per strategy

    for clf_name, clf in classifiers.items():
        steps = [
            ('smote',  smote),
            ('scaler', StandardScaler()),
        ]
        if selector is not None:
            steps.append(('pca', selector))
        steps.append(('clf', clf))

        pipeline = Pipeline(steps)
        t0 = time.time()

        try:
            scores = cross_validate(
                pipeline, X_sub, y,
                cv=cv,
                scoring=scoring,
                return_train_score=False,
                n_jobs=1,              # n_jobs=1: PCA wrapper is not thread-safe
                error_score='raise',
            )
            elapsed = time.time() - t0

            bal_acc = np.mean(scores['test_balanced_acc'])
            f1_val  = np.mean(scores['test_f1'])
            auc_val = np.mean(scores['test_roc_auc'])
            std_val = np.std (scores['test_balanced_acc'])

            results.append({
                'Feature Selection': strat_name,
                'Classifier':        clf_name,
                'N Features In':     len(feat_idx),
                'Balanced Acc':      bal_acc,
                'F1':                f1_val,
                'ROC-AUC':           auc_val,
                'Balanced Acc Std':  std_val,
                'Time (s)':          round(elapsed, 1),
            })
            print(f"  ✓  {strat_name:<20} | {clf_name:<16} | "
                  f"{bal_acc:.3f}  {f1_val:.3f}  {auc_val:.3f}  {elapsed:>5.1f}s")

        except Exception as e:
            elapsed = time.time() - t0
            print(f"  ✗  {strat_name:<20} | {clf_name:<16} | "
                  f"ERROR: {str(e).split(chr(10))[0][:55]}")
            results.append({
                'Feature Selection': strat_name,
                'Classifier':        clf_name,
                'N Features In':     len(feat_idx),
                'Balanced Acc': np.nan, 'F1': np.nan,
                'ROC-AUC': np.nan, 'Balanced Acc Std': np.nan,
                'Time (s)': round(elapsed, 1),
            })

# ── 6. Results table ───────────────────────────────────────────────────────────
df_results = (pd.DataFrame(results)
                .sort_values('Balanced Acc', ascending=False)
                .reset_index(drop=True))

print("\n" + "=" * 75)
print("COMBINED + PCA RESULTS  (sorted by Balanced Accuracy)")
print("=" * 75)
print(df_results.to_string(index=False, float_format='{:.3f}'.format))

# ── 7. Best combination ────────────────────────────────────────────────────────
best = df_results.dropna(subset=['Balanced Acc']).iloc[0]
print(f"\n🏆 Best combination:")
print(f"   Feature Selection : {best['Feature Selection']}")
print(f"   Classifier        : {best['Classifier']}")
print(f"   N Features (input): {int(best['N Features In'])}")
print(f"   Balanced Accuracy : {best['Balanced Acc']:.3f} "
      f"± {best['Balanced Acc Std']:.3f}")
print(f"   F1 Score          : {best['F1']:.3f}")
print(f"   ROC-AUC           : {best['ROC-AUC']:.3f}")

# ── 8. PCA component analysis (on full data, informational only) ──────────────
print("\n" + "=" * 75)
print("PCA COMPONENT ANALYSIS  (fitted on full scaled data — informational)")
print("=" * 75)

scaler_full = StandardScaler()

for label, idx in [('Grand Union (247 feats)', grand_idx),
                   ('Consensus  ( 39 feats)', consensus_idx)]:
    X_sc  = scaler_full.fit_transform(X[:, idx])
    pca_  = PCA(random_state=42).fit(X_sc)
    cumvar= np.cumsum(pca_.explained_variance_ratio_)
    n95   = np.searchsorted(cumvar, 0.95) + 1
    n99   = np.searchsorted(cumvar, 0.99) + 1
    print(f"\n  {label}:")
    print(f"    Components for 95% variance : {n95}")
    print(f"    Components for 99% variance : {n99}")
    print(f"    Variance explained by PC1   : "
          f"{pca_.explained_variance_ratio_[0]*100:.1f}%")
    print(f"    Variance explained by PC2   : "
          f"{pca_.explained_variance_ratio_[1]*100:.1f}%")
    print(f"    Variance explained by PC1+2 : "
          f"{cumvar[1]*100:.1f}%")

# ── 9. Save ────────────────────────────────────────────────────────────────────
df_results.to_csv('combined_pca_results.csv', index=False)
print("\n📄 combined_pca_results.csv saved")

# Save the grand union feature list in the same format as the other CSVs
from collections import Counter
all_method_sets = {
    'ANOVA': anova_feats, 'Mutual Info': mi_feats,
    'RFE-SVM': rfe_svm_feats, 'RFE-LogReg': rfe_lr_feats,
    'Boruta': boruta_feats,
    'Lasso': lasso_feats, 'ElasticNet': enet_feats,
    'RF-Embed': rf_feats, 'XGBoost': xgb_feats,
}
freq_counter = Counter()
for s in all_method_sets.values():
    freq_counter.update(s)

grand_df = pd.DataFrame({
    'feature':          sorted(grand_union, key=lambda f: int(f.split('_')[1])),
}).assign(
    n_methods       = lambda df: df['feature'].map(freq_counter),
    in_filter       = lambda df: df['feature'].isin(filter_all),
    in_wrapper      = lambda df: df['feature'].isin(wrapper_all),
    in_embedded     = lambda df: df['feature'].isin(embedded_all),
    in_consensus    = lambda df: df['feature'].isin(consensus_feats),
).sort_values('n_methods', ascending=False).reset_index(drop=True)

grand_df.to_csv('combined_features.csv', index=False)
print("📄 combined_features.csv saved  "
      "(247 features with per-method membership flags)")

COMBINED FEATURE SETS
  Filter   union  :   80 features
  Wrapper  union  :  161 features
  Embedded union  :  139 features
  Grand union     :  247 features  ← new 'Combined' set
  Consensus (×3)  :   39 features  (reference)

Strategy               | Classifier       |  BalAcc      F1     AUC    Time
---------------------------------------------------------------------------
  ✓  Combined (all)       | KNN              | 0.887  0.844  0.959    0.1s
  ✓  Combined (all)       | SVM              | 0.840  0.794  0.959    0.1s
  ✓  Combined (all)       | Naive Bayes      | 0.830  0.779  0.904    0.1s
  ✓  Combined (all)       | Decision Tree    | 0.665  0.557  0.665    0.1s
  ✓  Combined (all)       | Random Forest    | 0.830  0.767  0.918    0.9s
  ✓  Combined (all)       | Logistic Reg     | 0.893  0.859  0.970    0.1s
  ✓  Combined + PCA       | KNN              | 0.900  0.863  0.944    0.2s
  ✓  Combined + PCA       | SVM              | 0.880  0.843  0.964    0.2s
  ✓  Combined + PCA 

## Pca features combined 

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL — Filter ∩ Wrapper ∩ Embedded ∩ PCA Intersection
# ══════════════════════════════════════════════════════════════════════════════
# Finds which of the 39 consensus features are also the most structurally
# important according to PCA — i.e. which features dominate the principal
# components that explain 95% of the variance.
#
# PCA does not select features directly, so we use the SUM OF SQUARED LOADINGS
# of each feature across the kept components as a contribution score.
# Features that drive the PCA structure score high; redundant ones score low.
#
# Prerequisites: selected_features.csv, wrapper_selected_features.csv,
#                embedded_selected_features.csv, consensus_features.csv
#                X and y already defined in the notebook
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── 0. Re-load all 9 method feature sets ─────────────────────────────────────
filter_df   = pd.read_csv('selected_features.csv')
wrapper_df  = pd.read_csv('wrapper_selected_features.csv')
embedded_df = pd.read_csv('embedded_selected_features.csv')
consensus_df= pd.read_csv('consensus_features.csv')

anova_feats   = set(filter_df  [filter_df  ['selected_by_ANOVA']]    ['feature'])
mi_feats      = set(filter_df  [filter_df  ['selected_by_MI']]       ['feature'])
rfe_svm_feats = set(wrapper_df [wrapper_df ['selected_by_RFE_SVM']]  ['feature'])
rfe_lr_feats  = set(wrapper_df [wrapper_df ['selected_by_RFE_LogReg']]['feature'])
boruta_feats  = set(wrapper_df [wrapper_df ['selected_by_Boruta']]   ['feature'])
lasso_feats   = set(embedded_df[embedded_df['selected_by_Lasso']]    ['feature'])
enet_feats    = set(embedded_df[embedded_df['selected_by_ElasticNet']]['feature'])
rf_feats      = set(embedded_df[embedded_df['selected_by_RF']]       ['feature'])
xgb_feats     = set(embedded_df[embedded_df['selected_by_XGBoost']]  ['feature'])

# ── 1. Rebuild consensus = Filter ∩ Wrapper ∩ Embedded ───────────────────────
filter_all   = anova_feats   | mi_feats
wrapper_all  = rfe_svm_feats | rfe_lr_feats | boruta_feats
embedded_all = lasso_feats   | enet_feats   | rf_feats | xgb_feats
consensus    = filter_all & wrapper_all & embedded_all   # 39 features

# Sort by index so loadings align with columns
consensus_sorted = sorted(consensus, key=lambda f: int(f.split('_')[1]))
consensus_idx    = np.array([int(f.split('_')[1]) for f in consensus_sorted])

print("=" * 60)
print("INTERSECTION: Filter ∩ Wrapper ∩ Embedded ∩ PCA")
print("=" * 60)
print(f"  Consensus features (F ∩ W ∩ E) : {len(consensus)}")

# ── 2. Fit PCA on the 39 consensus features ───────────────────────────────────
X_con = X[:, consensus_idx]
X_sc  = StandardScaler().fit_transform(X_con)

pca   = PCA(random_state=42)
pca.fit(X_sc)

# How many components explain 95% of variance?
cumvar = np.cumsum(pca.explained_variance_ratio_)
n95    = np.searchsorted(cumvar, 0.95) + 1
print(f"  PCA components for 95% variance : {n95}")
print(f"  Variance explained by PC1       : {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"  Variance explained by PC1+PC2   : {cumvar[1]*100:.1f}%")

# ── 3. Compute feature contribution to the kept components ────────────────────
# pca.components_ shape: (n_components, n_features)
# Each value = loading of one feature on one component
# Sum of squared loadings across the kept n95 components = total contribution

top_components  = pca.components_[:n95]              # (n95, 39)
sum_sq_loadings = (top_components ** 2).sum(axis=0)  # (39,) one score per feature
max_loadings    = np.abs(top_components).max(axis=0) # (39,) peak loading

# ── 4. Build ranked loading table ─────────────────────────────────────────────
n_methods_map = consensus_df.set_index('feature')['n_individual_methods'].to_dict()

loading_df = pd.DataFrame({
    'feature':        consensus_sorted,
    'sum_sq_loading': sum_sq_loadings,
    'max_loading':    max_loadings,
    'pca_contrib_%':  (sum_sq_loadings / sum_sq_loadings.sum() * 100).round(2),
    'n_methods':      [n_methods_map[f] for f in consensus_sorted],
}).sort_values('sum_sq_loading', ascending=False).reset_index(drop=True)

loading_df['pca_rank'] = loading_df.index + 1

# ── 5. Determine the PCA-dominant features ────────────────────────────────────
# Keep features whose cumulative contribution reaches 95% of total loading mass
cumsum_contrib = (loading_df['sum_sq_loading'].cumsum() /
                  loading_df['sum_sq_loading'].sum())
n_pca_features = int((cumsum_contrib < 0.95).sum() + 1)

pca_dominant = set(loading_df.head(n_pca_features)['feature'])
excluded     = consensus - pca_dominant

print(f"  Features driving 95% of PCA    : {n_pca_features}")
print(f"  Features excluded by PCA       : {len(excluded)} → {sorted(excluded)}")

# ── 6. Final intersection ─────────────────────────────────────────────────────
intersection = consensus & pca_dominant   # = pca_dominant (subset of consensus)

print(f"\n  Filter ∩ Wrapper ∩ Embedded ∩ PCA = {len(intersection)} features")

# ── 7. Full results table ─────────────────────────────────────────────────────
result_df = loading_df.copy()
result_df['in_intersection'] = result_df['feature'].isin(intersection)

print("\n" + "=" * 60)
print("ALL 39 FEATURES — PCA contribution ranking")
print("(bottom 3 are excluded from the intersection)")
print("=" * 60)
print(result_df[['pca_rank', 'feature', 'pca_contrib_%',
                  'max_loading', 'n_methods', 'in_intersection'
                  ]].to_string(index=False, float_format='{:.3f}'.format))

# ── 8. The intersection features sorted by n_methods ─────────────────────────
print("\n" + "=" * 60)
print(f"THE {len(intersection)} INTERSECTION FEATURES")
print("sorted by n_individual_methods (trust score)")
print("=" * 60)
inter_df = result_df[result_df['in_intersection']].sort_values(
    'n_methods', ascending=False).reset_index(drop=True)
inter_df['rank'] = inter_df.index + 1
print(inter_df[['rank', 'feature', 'n_methods',
                 'pca_rank', 'pca_contrib_%']].to_string(index=False))

# ── 9. Save ───────────────────────────────────────────────────────────────────
result_df.to_csv('filter_wrapper_embedded_pca_intersection.csv', index=False)
print("\n📄 filter_wrapper_embedded_pca_intersection.csv saved")
print(f"   Columns: pca_rank | feature | pca_contrib_% | "
      f"max_loading | n_methods | in_intersection")

FileNotFoundError: [Errno 2] No such file or directory: 'selected_features.csv'